In [ ]:
# 1. 所有 packages 的 imports。先按随包说明准备兼容环境。
import math
import torch
from torch import nn
from torch.nn import functional as F
import csv
import hashlib
import json
from pathlib import Path
import cv2
import numpy as np
from mmcv.ops import box_iou_rotated
from PIL import Image, ImageDraw
from torch.utils.data import Dataset
import copy
import logging
from e2cnn.nn.modules.r2_conv import basisexpansion_singleblock as e2cnn_basis_cache
from mmcv import ConfigDict
from mmcv.ops import nms_rotated
from mmdet.core.bbox.assigners import MaxIoUAssigner
from mmdet.core.bbox.builder import BBOX_ASSIGNERS
from mmrotate.core import obb2xyxy
from mmrotate.models import build_detector
from mmrotate.models.builder import ROTATED_HEADS, ROTATED_DETECTORS
from mmrotate.models.roi_heads.roi_trans_roi_head import RoITransRoIHead
from mmrotate.models.detectors.redet import ReDet
import mmrotate.models.roi_heads.bbox_heads.rotated_bbox_head as rotated_bbox_head_module
import os
import platform
import random
import time
import urllib.request
import uuid
import mmcv
import mmdet
import mmrotate
from torch.utils.data import DataLoader

In [ ]:
# 2. 总控制阀：只在这里设置数据、模式、训练、模型和推理参数。
# train/resume: 带 OBB 标注及病例组的 PNG/JPG/JPEG；infer: 图片目录 + 本模型训练权重。
# AU-OPG classes: Cavitated, Crowned, Filled, Impacted, Implant, Retained Root。
# OCD classes: cell。JSON manifest可提供类别顺序；DOTA目录请填写classes。
CFG = {'mode': 'train',
 'manifest_path': '',
 'data_root': './data',
 'inference_dir': './images',
 'output_dir': './srd_results',
 'checkpoint_path': '',
 'classes': [],
 'allow_image_groups': False,
 'train_horizontal_flip_prob': 0.25,
 'train_vertical_flip_prob': 0.25,
 'image_size': 1024,
 'image_mean': [123.675, 116.28, 103.53],
 'image_std': [58.395, 57.12, 57.375],
 'device': 'auto',
 'seed': 0,
 'num_workers': 0,
 'batch_size': 1,
 'accumulation_steps': 2,
 'num_threads': 4,
 'amp': False,
 'pretrained_backbone': '',
 'pretrained_frozen_stages': 1,
 'stage_a_epochs': 30,
 'stage_b_epochs': 20,
 'optimizer': 'SGD',
 'learning_rate_a': 0.0025,
 'learning_rate_b': 0.00025,
 'momentum': 0.9,
 'weight_decay': 0.0001,
 'gradient_clip': 35.0,
 'warmup_steps': 500,
 'warmup_ratio': 0.3333333333333333,
 'lr_decay_milestones': [0.66, 0.92],
 'lr_decay_gamma': 0.1,
 'early_stopping_patience': 4,
 'early_stopping_start_epoch': 10,
 'early_stopping_min_delta': 0.001,
 'monitor_metric': 'mAP50:95',
 'evaluate_test': True,
 'save_overlays': True,
 'score_threshold': 0.05,
 'max_detections': 2000,
 'metric_score_threshold': 0.25,
 'metric_match_iou': 0.5,
 'checkpoint_every': 1,
 'validation_interval': 1,
 'deterministic': False,
 'bank_max_instances_per_chunk': 256,
 'reasoning': {'channels': 256,
               'key_dim': 32,
               'evidence_dim': 192,
               'projection_seed': 20261016,
               'evidence_layer_scale': 0.001,
               'spectral_enabled': True,
               'memory_enabled': True,
               'mrf_enabled': True,
               'spectral': {'reduced_channels': 64,
                            'max_rois': 128,
                            'max_fraction': 0.5,
                            'risk_threshold': 0.25,
                            'small_object_scale': 64.0,
                            'entropy_weight': 0.45,
                            'size_weight': 0.3,
                            'direction_weight': 0.25,
                            'spectral_conflict_weight': 0.5,
                            'anisotropy_floor': 0.05,
                            'gate_slope': 4.0,
                            'layerscale_init': 0.001,
                            'frequency_init_std': 0.02,
                            'numerical_eps': 1e-06},
               'memory': {'num_classes': 6,
                          'prior_mean': 0.0,
                          'kappa0': 1.0,
                          'alpha0': 2.0,
                          'beta0': 1.0,
                          'standardization_floor': 0.05,
                          'epsilon': 0.3,
                          'null_cost': 2.0,
                          'null_prior': 0.25,
                          'capacity_enabled': True,
                          'capacity_factor': 1.25,
                          'dual_steps': 80,
                          'solver_tolerance': 0.0001,
                          'value_norm_clip': 1.0,
                          'numerical_eps': 1e-08},
               'mrf': {'unary_precision': 1.0,
                       'mu_floor': 0.2,
                       'lambda_pairwise': 0.05,
                       'eta_memory': 0.1,
                       'rho_delta': 0.1,
                       'inner_steps': 5,
                       'neighbors': 8,
                       'position_scale': 3.0,
                       'feature_temperature': 0.5,
                       'weight_max': 1.0,
                       'numerical_eps': 1e-08}},
 'model': {'type': 'ReDet',
           'backbone': {'type': 'ReResNet',
                        'depth': 50,
                        'num_stages': 4,
                        'out_indices': (0, 1, 2, 3),
                        'frozen_stages': -1,
                        'style': 'pytorch',
                        'pretrained': None},
           'neck': {'type': 'ReFPN',
                    'in_channels': [256, 512, 1024, 2048],
                    'out_channels': 256,
                    'num_outs': 5},
           'rpn_head': {'type': 'RotatedRPNHead',
                        'in_channels': 256,
                        'feat_channels': 256,
                        'version': 'le90',
                        'anchor_generator': {'type': 'AnchorGenerator',
                                             'scales': [8],
                                             'ratios': [0.5, 1.0, 2.0],
                                             'strides': [4, 8, 16, 32, 64]},
                        'bbox_coder': {'type': 'DeltaXYWHBBoxCoder',
                                       'target_means': [0.0, 0.0, 0.0, 0.0],
                                       'target_stds': [1.0, 1.0, 1.0, 1.0]},
                        'loss_cls': {'type': 'CrossEntropyLoss', 'use_sigmoid': True, 'loss_weight': 1.0},
                        'loss_bbox': {'type': 'SmoothL1Loss',
                                      'beta': 0.1111111111111111,
                                      'loss_weight': 1.0}},
           'roi_head': {'type': 'RoITransRoIHead',
                        'version': 'le90',
                        'num_stages': 2,
                        'stage_loss_weights': [1, 1],
                        'bbox_roi_extractor': [{'type': 'SingleRoIExtractor',
                                                'roi_layer': {'type': 'RoIAlign',
                                                              'output_size': 7,
                                                              'sampling_ratio': 0},
                                                'out_channels': 256,
                                                'featmap_strides': [4, 8, 16, 32]},
                                               {'type': 'RotatedSingleRoIExtractor',
                                                'roi_layer': {'type': 'RiRoIAlignRotated',
                                                              'out_size': 7,
                                                              'num_samples': 2,
                                                              'num_orientations': 8,
                                                              'clockwise': True},
                                                'out_channels': 256,
                                                'featmap_strides': [4, 8, 16, 32]}],
                        'bbox_head': [{'type': 'RotatedShared2FCBBoxHead',
                                       'in_channels': 256,
                                       'fc_out_channels': 1024,
                                       'roi_feat_size': 7,
                                       'num_classes': 1,
                                       'bbox_coder': {'type': 'DeltaXYWHAHBBoxCoder',
                                                      'angle_range': 'le90',
                                                      'norm_factor': 2,
                                                      'edge_swap': True,
                                                      'target_means': [0.0, 0.0, 0.0, 0.0, 0.0],
                                                      'target_stds': [0.1, 0.1, 0.2, 0.2, 0.1]},
                                       'reg_class_agnostic': True,
                                       'loss_cls': {'type': 'CrossEntropyLoss',
                                                    'use_sigmoid': False,
                                                    'loss_weight': 1.0},
                                       'loss_bbox': {'type': 'SmoothL1Loss',
                                                     'beta': 1.0,
                                                     'loss_weight': 1.0}},
                                      {'type': 'RotatedShared2FCBBoxHead',
                                       'in_channels': 256,
                                       'fc_out_channels': 1024,
                                       'roi_feat_size': 7,
                                       'num_classes': 1,
                                       'bbox_coder': {'type': 'DeltaXYWHAOBBoxCoder',
                                                      'angle_range': 'le90',
                                                      'norm_factor': None,
                                                      'edge_swap': True,
                                                      'proj_xy': True,
                                                      'target_means': [0.0, 0.0, 0.0, 0.0, 0.0],
                                                      'target_stds': [0.05, 0.05, 0.1, 0.1, 0.05]},
                                       'reg_class_agnostic': False,
                                       'loss_cls': {'type': 'CrossEntropyLoss',
                                                    'use_sigmoid': False,
                                                    'loss_weight': 1.0},
                                       'loss_bbox': {'type': 'SmoothL1Loss',
                                                     'beta': 1.0,
                                                     'loss_weight': 1.0}}]},
           'train_cfg': {'rpn': {'assigner': {'type': 'MaxIoUAssigner',
                                              'pos_iou_thr': 0.7,
                                              'neg_iou_thr': 0.3,
                                              'min_pos_iou': 0.3,
                                              'match_low_quality': True,
                                              'ignore_iof_thr': -1},
                                 'sampler': {'type': 'RandomSampler',
                                             'num': 256,
                                             'pos_fraction': 0.5,
                                             'neg_pos_ub': -1,
                                             'add_gt_as_proposals': False},
                                 'allowed_border': 0,
                                 'pos_weight': -1,
                                 'debug': False},
                         'rpn_proposal': {'nms_pre': 2000,
                                          'max_per_img': 2000,
                                          'nms': {'type': 'nms', 'iou_threshold': 0.7},
                                          'min_bbox_size': 0},
                         'rcnn': [{'assigner': {'type': 'MaxIoUAssigner',
                                                'pos_iou_thr': 0.5,
                                                'neg_iou_thr': 0.5,
                                                'min_pos_iou': 0.5,
                                                'match_low_quality': False,
                                                'ignore_iof_thr': -1,
                                                'iou_calculator': {'type': 'BboxOverlaps2D'}},
                                   'sampler': {'type': 'RandomSampler',
                                               'num': 512,
                                               'pos_fraction': 0.25,
                                               'neg_pos_ub': -1,
                                               'add_gt_as_proposals': True},
                                   'pos_weight': -1,
                                   'debug': False},
                                  {'assigner': {'type': 'MaxIoUAssigner',
                                                'pos_iou_thr': 0.5,
                                                'neg_iou_thr': 0.5,
                                                'min_pos_iou': 0.5,
                                                'match_low_quality': False,
                                                'ignore_iof_thr': -1,
                                                'iou_calculator': {'type': 'RBboxOverlaps2D'}},
                                   'sampler': {'type': 'RRandomSampler',
                                               'num': 512,
                                               'pos_fraction': 0.25,
                                               'neg_pos_ub': -1,
                                               'add_gt_as_proposals': True},
                                   'pos_weight': -1,
                                   'debug': False}]},
           'test_cfg': {'rpn': {'nms_pre': 2000,
                                'max_per_img': 2000,
                                'nms': {'type': 'nms', 'iou_threshold': 0.7},
                                'min_bbox_size': 0},
                        'rcnn': {'nms_pre': 2000,
                                 'min_bbox_size': 0,
                                 'score_thr': 0.05,
                                 'nms': {'iou_thr': 0.1},
                                 'max_per_img': 2000}}},
 'groups_csv': None,
 'annotation_bounds_tolerance': 1.0,
 'eval_device': 'cpu',
 'overlay_score_threshold': 0.5,
 'overlay_line_width': 2,
 'download_pretrained': True,
 'pretrained_url': 'https://drive.usercontent.google.com/download?id=1FshfREfLZaNl5FcaKrH0lxFyZt50Uyu2&export=download&authuser=0&confirm=t',
 'pretrained_sha256': '25b168462b03f67c9549449cd0dbf282c9a801ccd3888e2310329148beb25ce2',
 'adam_betas': [0.9, 0.999],
 'adam_eps': 1e-08,
 'ignore_iof_threshold': 0.5,
 'inference_overrides': {},
 'evaluation_split': 'test'}

# SRD：参考证据驱动的二维旋转框检测

一个模型：官方 ReDet C8/两级 RoI 几何 + 条件频域残差 + NIG/Student-t 训练记忆 + 固定凸证据场。
先修改第 2 格配置，再按顺序运行所有格。

**第一次训练**：`mode='train'`，设置 `manifest_path` 或 DOTA `data_root`，使用新 `output_dir`。
**继续训练**：`mode='resume'`，`checkpoint_path` 指向 `stage_A_last.pth` / `stage_B_last.pth`；保留对应 best 文件。
**图片推理**：`mode='infer'`，设置 `inference_dir` 和训练产生的 `stage_B_best.pth`。模型结构、类别及预处理从权重恢复。
**单独评估**：`mode='evaluate'`，提供标注及 `evaluation_split`。

默认训练 A 30 + B 20 epochs，B 冻结骨干/FPN/RPN/首级检测头。参考库仅由 best A 的训练组构建，B 查询排除同组，部署时不更新。

### 标注例（manifest.json）
```json
{"classes":["cell"],"records":[
 {"id":"train01","image":"images/train01.png","split":"train","group_id":"patient01","reviewed":true,"boxes":[[100,80,40,15,0.3]],"labels":[0]},
 {"id":"val01","image":"images/val01.jpg","split":"val","group_id":"patient02","reviewed":true,"boxes":[[70,60,30,10,-0.2]],"labels":[0]}
]}
```
`boxes` 为原图像素坐标 `[cx,cy,w,h,theta]`，顺时针弧度、le90；也支持四点 `polygons`。
组标识不得跨 train/val/test；跨划分重复像素会报错。完整数据规范与证据见随包 README。

In [ ]:
# reasoning
"""Conditional spectral evidence, frozen-space Bayesian memory and convex graph inference.

All modules are self-contained PyTorch code. Bayesian claims apply to the fixed key
space only; the graph guarantee concerns its fixed-coefficient evidence energy.
"""







def nig_posterior(observations, prior_mean=0., kappa0=1., alpha0=2., beta0=1.):
    """Conjugate diagonal Normal-Inverse-Gamma update from independent observations."""
    n = observations.shape[0]
    base = observations.new_tensor(prior_mean)
    kappa = observations.new_tensor(kappa0 + n)
    alpha = observations.new_tensor(alpha0 + n / 2.)
    if n == 0:
        shape = observations.shape[1:]
        return base.expand(shape), kappa, alpha, observations.new_full(shape, beta0)
    average = observations.mean(0)
    scatter = (observations - average).square().sum(0)
    mean = (kappa0 * base + n * average) / kappa
    beta = beta0 + 0.5 * scatter + kappa0 * n * (average - base).square() / (2. * kappa)
    return mean, kappa, alpha, beta


def student_t_log_prob(z, mean, kappa, alpha, beta):
    """Full predictive log density, including scale and normalization terms."""
    nu = 2. * alpha
    scale_sq = beta * (kappa + 1.) / (alpha * kappa)
    return (torch.lgamma((nu + 1.) / 2.) - torch.lgamma(nu / 2.)
            - 0.5 * (torch.log(nu * math.pi) + torch.log(scale_sq))
            - ((nu + 1.) / 2.) * torch.log1p((z - mean).square() / (nu * scale_sq)))


class SpectralRoIAdapter(nn.Module):
    """Vectorized FFT residual on a bounded subset of already aligned RoIs.

    Geometry uses [batch,cx,cy,w,h,theta]. Directions are axial and expressed in
    the canonical RoI frame. This adapter does not claim continuous equivariance.
    The structure tensor and the spectrum are different observed feature cues,
    not regression uncertainty or independently sampled Bayesian evidence.
    """
    DEFAULTS = {
        "reduced_channels": 64, "max_rois": 128, "max_fraction": 0.5,
        "risk_threshold": 0.25, "small_object_scale": 64.,
        "entropy_weight": 0.45, "size_weight": 0.3, "direction_weight": 0.25,
        "spectral_conflict_weight": 0.5, "anisotropy_floor": 0.05,
        "gate_slope": 4., "layerscale_init": 1e-3,
        "frequency_init_std": 0.02, "numerical_eps": 1e-6,
    }

    def __init__(self, channels, cfg=None):
        super().__init__()
        self.cfg = dict(self.DEFAULTS, **(cfg or {}))
        reduced = int(self.cfg["reduced_channels"])
        if channels < 1 or reduced < 1 or int(self.cfg["max_rois"]) < 0:
            raise ValueError("Channels must be positive and max_rois nonnegative")
        if not 0 <= self.cfg["max_fraction"] <= 1:
            raise ValueError("max_fraction must lie in [0,1]")
        self.reduce = nn.Conv2d(channels, reduced, 1, bias=False)
        self.expand = nn.Conv2d(reduced, channels, 1, bias=False)
        self.frequency_response = nn.Parameter(torch.empty(reduced, 3))
        nn.init.normal_(self.frequency_response, std=self.cfg["frequency_init_std"])
        self.layerscale = nn.Parameter(torch.full((1, channels, 1, 1), self.cfg["layerscale_init"]))

    def _cheap_risk(self, feats, rois, class_prob, orientation_hint):
        eps = self.cfg["numerical_eps"]
        # Channel RMS avoids cancellation of signed appearance features.
        signal = (feats.detach().float().square().mean(1) + eps).sqrt()
        dx = signal[:, 1:-1, 2:] - signal[:, 1:-1, :-2]
        dy = signal[:, 2:, 1:-1] - signal[:, :-2, 1:-1]
        xx, yy, xy = dx.square().mean((1, 2)), dy.square().mean((1, 2)), (dx * dy).mean((1, 2))
        norm = ((xx - yy).square() + 4. * xy.square() + eps ** 2).sqrt() - eps
        anisotropy = (norm / (xx + yy + eps)).clamp(0., 1.)
        # The spatial long axis is perpendicular to the dominant gradient axis.
        cos2 = -(xx - yy) / (norm + eps)
        if orientation_hint is not None:
            hint = orientation_hint.detach().float()
            if hint.ndim == 1:
                cos2 = torch.cos(2. * hint)
            elif hint.shape[1] == 2:
                magnitude = hint.norm(dim=1).clamp_min(eps)
                cos2 = hint[:, 0] / magnitude
                anisotropy = magnitude.clamp(0., 1.)
            else:
                # Optional orientation energies are already in the local frame.
                probabilities = hint.clamp_min(0.)
                probabilities = probabilities / probabilities.sum(1, keepdim=True).clamp_min(eps)
                angle = torch.arange(hint.shape[1], device=hint.device) * (2. * math.pi / hint.shape[1])
                axial_x = (probabilities * torch.cos(2. * angle)).sum(1)
                axial_y = (probabilities * torch.sin(2. * angle)).sum(1)
                anisotropy = (axial_x.square() + axial_y.square()).sqrt().clamp(0., 1.)
                cos2 = axial_x / anisotropy.clamp_min(eps)
        aspect = (rois[:, 3] - rois[:, 4]).abs() / (rois[:, 3] + rois[:, 4]).clamp_min(eps)
        directional_conflict = 0.5 * (1. - cos2.clamp(-1., 1.)) * aspect * anisotropy
        directional_conflict = directional_conflict * (anisotropy >= self.cfg["anisotropy_floor"])
        if class_prob is None:
            entropy = signal.new_zeros(signal.shape[0])
        else:
            p = class_prob.detach().float().clamp_min(eps)
            if p.ndim != 2 or p.shape[0] != len(feats) or p.shape[1] < 2:
                raise ValueError("class_prob must include foreground AND background classes")
            p = p / p.sum(1, keepdim=True)
            entropy = -(p * p.log()).sum(1) / math.log(p.shape[1])
        size = (self.cfg["small_object_scale"] / (rois[:, 3] * rois[:, 4]).clamp_min(eps).sqrt()).clamp(max=1.)
        risk = (self.cfg["entropy_weight"] * entropy + self.cfg["size_weight"] * size
                + self.cfg["direction_weight"] * directional_conflict)
        return risk, anisotropy

    def forward(self, feats, rois, class_prob=None, orientation_hint=None):
        if feats.ndim != 4 or feats.shape[2] < 3 or feats.shape[3] < 3:
            raise ValueError("Expected [N,C,H,W] aligned RoIs with H,W >= 3")
        if rois.shape != (feats.shape[0], 6):
            raise ValueError("RoIs must have shape [N,6]")
        n = feats.shape[0]
        selected = torch.zeros(n, device=feats.device, dtype=torch.bool)
        empty = feats.new_zeros(n, dtype=torch.float32)
        diag = {"selected_mask": selected, "selected_indices": selected.nonzero().flatten(),
                "fft_rois": 0, "cheap_risk": empty, "feature_anisotropy": empty,
                "spectral_anisotropy": empty, "spectral_conflict": empty,
                "direction_source": "canonical_roi_structure_tensor"}
        if n == 0:
            return feats, diag
        risk, feature_anisotropy = self._cheap_risk(feats, rois.float(), class_prob, orientation_hint)
        # Budgets are per image: unrelated batch members cannot alter selection.
        for image in torch.unique(rois[:, 0]):
            image_mask = rois[:, 0] == image
            count = int(image_mask.sum())
            budget = min(int(self.cfg["max_rois"]), int(math.ceil(count * self.cfg["max_fraction"])))
            candidates = torch.nonzero(image_mask & (risk >= self.cfg["risk_threshold"])).flatten()
            if budget > 0 and candidates.numel():
                chosen = candidates[torch.topk(risk[candidates], min(budget, len(candidates)), sorted=False).indices]
                selected[chosen] = True
        indices = torch.nonzero(selected).flatten()
        diag.update(selected_indices=indices, fft_rois=len(indices), cheap_risk=risk.detach(),
                    feature_anisotropy=feature_anisotropy.detach())
        if not len(indices):
            return feats, diag
        reduced = self.reduce(feats[indices]).float()
        height, width = reduced.shape[-2:]
        spectrum = torch.fft.rfft2(reduced, norm="ortho")
        fy = torch.fft.fftfreq(height, device=feats.device).view(height, 1)
        fx = torch.fft.rfftfreq(width, device=feats.device).view(1, width // 2 + 1)
        radius2 = fx.square() + fy.square()
        eps = self.cfg["numerical_eps"]
        basis = torch.stack((radius2 / radius2.max().clamp_min(eps),
                             (fx.square() - fy.square()) / radius2.clamp_min(eps),
                             2. * fx * fy / radius2.clamp_min(eps)))
        # An even real frequency response preserves the real-valued FFT contract.
        response = torch.tanh(torch.einsum("ck,khw->chw", self.frequency_response.float(), basis))
        response = response * (radius2 > 0.)  # DC remains on the unchanged main path.
        correction = torch.fft.irfft2(spectrum * response[None], s=(height, width), norm="ortho")
        # rFFT interior columns represent both positive and negative frequencies.
        multiplicity = torch.full_like(fx, 2.)
        multiplicity[:, 0] = 1.
        if width % 2 == 0:
            multiplicity[:, -1] = 1.
        power = spectrum.abs().square().mean(1) * multiplicity
        xx, yy, xy = (power * fx.square()).sum((1, 2)), (power * fy.square()).sum((1, 2)), (power * fx * fy).sum((1, 2))
        norm = ((xx - yy).square() + 4. * xy.square() + eps ** 2).sqrt() - eps
        anisotropy = (norm / (xx + yy + eps)).clamp(0., 1.)
        spatial_cos2 = -(xx - yy) / (norm + eps)
        aspect = (rois[indices, 3] - rois[indices, 4]).abs() / (rois[indices, 3] + rois[indices, 4]).clamp_min(eps)
        conflict = 0.5 * (1. - spatial_cos2.clamp(-1., 1.)) * anisotropy * aspect
        conflict = conflict * (anisotropy >= self.cfg["anisotropy_floor"])
        gate = torch.sigmoid(self.cfg["gate_slope"] * (risk[indices] - self.cfg["risk_threshold"])
                             + self.cfg["spectral_conflict_weight"] * conflict)
        residual = self.expand(correction.to(feats.dtype)) * self.layerscale * gate[:, None, None, None]
        output = feats.clone()
        output[indices] = feats[indices] + residual.to(feats.dtype)
        diag["spectral_anisotropy"] = empty.clone().index_copy(0, indices, anisotropy.detach())
        diag["spectral_conflict"] = empty.clone().index_copy(0, indices, conflict.detach())
        return output, diag


class BayesianMemory(nn.Module):
    """Train-only, independent-group NIG prototypes with null/capacity reading.

    A set_bank call freezes detached representatives. Forward never updates them.
    Training query groups are excluded BEFORE standardization and posterior fit.
    An unconverged finite dual solve is reported; any excess column mass is
    conservatively transferred to null, producing a feasible approximate read.
    """
    DEFAULTS = {
        "num_classes": 6, "prior_mean": 0., "kappa0": 1., "alpha0": 2., "beta0": 1.,
        "standardization_floor": 0.05, "epsilon": 0.3, "null_cost": 2.,
        "null_prior": 0.25, "capacity_enabled": True, "capacity_factor": 1.25,
        "dual_steps": 80, "solver_tolerance": 1e-4, "value_norm_clip": 1.,
        "numerical_eps": 1e-8,
    }

    def __init__(self, key_dim=32, value_dim=192, cfg=None):
        super().__init__()
        self.cfg = dict(self.DEFAULTS, **(cfg or {}))
        self.key_dim, self.value_dim = int(key_dim), int(value_dim)
        if min(self.key_dim, self.value_dim, self.cfg["num_classes"]) < 1:
            raise ValueError("Dimensions and class count must be positive")
        if min(self.cfg[k] for k in ("kappa0", "alpha0", "beta0", "epsilon", "standardization_floor", "value_norm_clip")) <= 0:
            raise ValueError("NIG, temperature, standardization and norm parameters must be positive")
        if not 0 < self.cfg["null_prior"] < 1 or self.cfg["capacity_factor"] < 0:
            raise ValueError("Invalid null prior or capacity factor")
        self.register_buffer("bank_keys", torch.empty(0, self.key_dim))
        self.register_buffer("bank_values", torch.empty(0, self.value_dim))
        self.register_buffer("bank_labels", torch.empty(0, dtype=torch.long))
        self.register_buffer("bank_groups", torch.empty(0, dtype=torch.long))

    def set_bank(self, keys, values, labels, groups, source_split="train"):
        if source_split != "train":
            raise ValueError("The reference bank accepts training representatives only")
        if keys.ndim != 2 or keys.shape[1] != self.key_dim or values.shape != (len(keys), self.value_dim):
            raise ValueError("Bank key/value dimensions do not match")
        if labels.shape != (len(keys),) or groups.shape != (len(keys),):
            raise ValueError("Bank labels and groups must be one-dimensional")
        if labels.is_floating_point() or groups.is_floating_point():
            raise ValueError("Labels/groups must use integer identifiers")
        if not torch.isfinite(keys).all() or not torch.isfinite(values).all():
            raise ValueError("Bank features must be finite")
        if len(labels) and (labels.min() < 0 or labels.max() >= self.cfg["num_classes"]):
            raise ValueError("Bank labels fall outside configured foreground ontology")
        device = self.bank_keys.device
        keys, values = keys.detach().float().to(device), values.detach().float().to(device)
        labels, groups = labels.detach().long().to(device), groups.detach().long().to(device)
        if len(keys):
            identities, inverse = torch.unique(torch.stack((labels, groups), 1), dim=0, return_inverse=True)
            count = torch.bincount(inverse, minlength=len(identities)).to(keys.dtype)[:, None]
            aggregate_keys = torch.zeros(len(identities), self.key_dim, device=device).index_add(0, inverse, keys) / count
            aggregate_values = torch.zeros(len(identities), self.value_dim, device=device).index_add(0, inverse, values) / count
            labels, groups = identities[:, 0], identities[:, 1]
            keys, values = aggregate_keys, aggregate_values
        self.bank_keys, self.bank_values = keys, values
        self.bank_labels, self.bank_groups = labels, groups

    def export_bank(self):
        return {"schema_version": 1, "source_split": "train", "key_dim": self.key_dim,
                "value_dim": self.value_dim, "num_classes": self.cfg["num_classes"],
                "keys": self.bank_keys.detach().cpu().clone(), "values": self.bank_values.detach().cpu().clone(),
                "labels": self.bank_labels.detach().cpu().clone(), "groups": self.bank_groups.detach().cpu().clone()}

    def import_bank(self, payload):
        if payload.get("schema_version") != 1 or payload.get("source_split") != "train":
            raise ValueError("Unrecognized or nontraining bank")
        if (payload.get("key_dim"), payload.get("value_dim"), payload.get("num_classes")) != (self.key_dim, self.value_dim, self.cfg["num_classes"]):
            raise ValueError("Bank dimensions or class ontology differ")
        self.set_bank(payload["keys"], payload["values"], payload["labels"], payload["groups"])

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):
        # Dynamic banks must resize before Module's ordinary checked copy.
        for name in ("bank_keys", "bank_values", "bank_labels", "bank_groups"):
            key = prefix + name
            if key in state_dict:
                setattr(self, name, torch.empty_like(state_dict[key], device=self.bank_keys.device))
        super()._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)

    def _read_pool(self, queries, keep):
        refs, vals, labels = self.bank_keys[keep], self.bank_values[keep], self.bank_labels[keep]
        n, device = len(queries), queries.device
        if not len(refs):
            zeros = queries.new_zeros(n)
            return queries.new_zeros(n, self.value_dim), zeros, {
                "null_mass": torch.ones_like(zeros), "capacity_violation": queries.new_zeros(()),
                "raw_capacity_violation": queries.new_zeros(()), "dual_residual": queries.new_zeros(()),
                "feasibility_repaired": False, "converged": True, "row_mass_error": queries.new_zeros(())}
        mean = refs.mean(0)
        std = refs.std(0, unbiased=False).clamp_min(self.cfg["standardization_floor"])
        standardized_refs, standardized_query = (refs - mean) / std, (queries - mean) / std
        present = torch.unique(labels, sorted=True)
        costs, values, counts = [], [], []
        for label in present:
            selected = labels == label
            posterior = nig_posterior(standardized_refs[selected], self.cfg["prior_mean"],
                                      self.cfg["kappa0"], self.cfg["alpha0"], self.cfg["beta0"])
            costs.append(-student_t_log_prob(standardized_query, *posterior).mean(1))
            value = vals[selected].mean(0)
            values.append(value * (self.cfg["value_norm_clip"] / value.norm().clamp_min(self.cfg["numerical_eps"])).clamp(max=1.))
            counts.append(selected.sum())
        cost = torch.stack(costs, 1)
        value = torch.stack(values)
        count = torch.stack(counts).float()
        class_prior = count / count.sum()
        prior = torch.cat((cost.new_tensor([self.cfg["null_prior"]]), (1. - self.cfg["null_prior"]) * class_prior))
        cost = torch.cat((cost.new_full((n, 1), self.cfg["null_cost"]), cost), 1)
        capacity = self.cfg["capacity_factor"] * class_prior
        dual = cost.new_zeros(len(present))
        epsilon = self.cfg["epsilon"]
        steps = int(self.cfg["dual_steps"]) if self.cfg["capacity_enabled"] else 0
        # Row masses sum to one; epsilon is a conservative dual step size.
        for _ in range(steps):
            adjusted = cost + torch.cat((dual.new_zeros(1), dual))[None]
            probabilities = torch.softmax(prior.log()[None] - adjusted / epsilon, dim=1)
            dual = torch.relu(dual + epsilon * (probabilities[:, 1:].mean(0) - capacity))
        adjusted = cost + torch.cat((dual.new_zeros(1), dual))[None]
        probabilities = torch.softmax(prior.log()[None] - adjusted / epsilon, dim=1)
        real = probabilities[:, 1:]
        if self.cfg["capacity_enabled"]:
            mass = real.mean(0)
            raw_violation = (mass - capacity).clamp_min(0.).max()
            residual = (dual - torch.relu(dual + epsilon * (mass - capacity))).abs().max() / epsilon
            # Conservative feasible fallback preserves row mass by sending excess to null.
            shrink = (capacity / mass.clamp_min(self.cfg["numerical_eps"])).clamp(max=1.)
            real = real * shrink[None]
            violation = (real.mean(0) - capacity).clamp_min(0.).max()
            repaired = bool((raw_violation > self.cfg["solver_tolerance"]).detach())
            converged = bool((residual <= self.cfg["solver_tolerance"]).detach())
        else:
            raw_violation = residual = violation = cost.new_zeros(())
            repaired, converged = False, True
        reliability = real.sum(1)
        null = 1. - reliability
        result = real @ value  # Do not renormalize the real columns.
        diag = {"null_mass": null.detach(), "capacity_violation": violation.detach(),
                "raw_capacity_violation": raw_violation.detach(), "dual_residual": residual.detach(),
                "feasibility_repaired": repaired, "converged": converged,
                "row_mass_error": (real.sum(1) + null - 1.).abs().max().detach()}
        return result, reliability, diag

    def forward(self, query_keys, query_groups=None, query_image_ids=None):
        if query_keys.ndim != 2 or query_keys.shape[1] != self.key_dim:
            raise ValueError("Unexpected query key dimension")
        if query_keys.device != self.bank_keys.device:
            raise ValueError("Move BayesianMemory to the query device before reading")
        if not torch.isfinite(query_keys).all():
            raise ValueError("Query keys must be finite")
        n = len(query_keys)
        if query_groups is not None and (query_groups.shape != (n,) or query_groups.is_floating_point()):
            raise ValueError("query_groups must be integer identifiers of shape [N]")
        if query_image_ids is not None and (query_image_ids.shape != (n,) or query_image_ids.is_floating_point()):
            raise ValueError("query_image_ids must be integer identifiers of shape [N]")
        query_keys = query_keys.float()
        result = query_keys.new_zeros(n, self.value_dim)
        reliability = query_keys.new_zeros(n)
        null = query_keys.new_ones(n)
        retained = torch.zeros(n, device=query_keys.device, dtype=torch.long)
        maximums = {name: query_keys.new_zeros(()) for name in (
            "capacity_violation", "raw_capacity_violation", "dual_residual", "row_mass_error")}
        all_converged, any_repaired = True, False
        image_ids = torch.zeros(n, device=query_keys.device, dtype=torch.long) if query_image_ids is None else query_image_ids.to(query_keys.device)
        group_ids = torch.zeros(n, device=query_keys.device, dtype=torch.long) if query_groups is None else query_groups.to(query_keys.device)
        partitions = torch.unique(torch.stack((image_ids, group_ids), 1), dim=0)
        for image, group in partitions:
            select = (image_ids == image) & (group_ids == group)
            if not select.any():
                continue
            keep = torch.ones(len(self.bank_keys), device=query_keys.device, dtype=torch.bool) if query_groups is None else self.bank_groups != group
            read, strength, diag = self._read_pool(query_keys[select], keep)
            result[select], reliability[select], null[select] = read, strength, diag["null_mass"]
            retained[select] = torch.unique(self.bank_groups[keep]).numel()
            for name in maximums:
                maximums[name] = torch.maximum(maximums[name], diag[name])
            all_converged = all_converged and diag["converged"]
            any_repaired = any_repaired or diag["feasibility_repaired"]
        return result, reliability, dict(maximums, null_mass=null, retained_group_counts=retained,
            converged=all_converged, feasibility_repaired=any_repaired,
            posterior_scope="fixed_key_diagonal_NIG", reference_count=len(self.bank_keys))


class SmoothTVReasoner(nn.Module):
    """Strongly convex smooth-TV refinement in unbounded RoI evidence space.

    The memory residual enters ONLY the soft target h0 + residual. Weights and
    graph are formed once per call and remain fixed throughout the inner solve.
    No operation smooths RoI coordinates or merges candidate identities.
    """
    DEFAULTS = {"unary_precision": 1., "mu_floor": 0.2, "lambda_pairwise": 0.05,
                "eta_memory": 0.1, "rho_delta": 0.1, "inner_steps": 5,
                "neighbors": 8, "position_scale": 3., "feature_temperature": 0.5,
                "weight_max": 1., "numerical_eps": 1e-8}

    def __init__(self, cfg=None):
        super().__init__()
        self.cfg = dict(self.DEFAULTS, **(cfg or {}))
        if self.cfg["unary_precision"] < self.cfg["mu_floor"] or self.cfg["mu_floor"] <= 0:
            raise ValueError("The unary precision must preserve a positive strong-convexity floor")
        if min(self.cfg[k] for k in ("rho_delta", "position_scale", "feature_temperature")) <= 0:
            raise ValueError("Smoothing/scale parameters must be positive")
        if min(self.cfg[k] for k in ("lambda_pairwise", "eta_memory", "weight_max", "inner_steps", "neighbors")) < 0:
            raise ValueError("Graph coefficients and budgets must be nonnegative")

    def _graph(self, rois, keys):
        edges = []
        n = len(rois)
        neighbors = int(self.cfg["neighbors"])
        if n < 2 or neighbors == 0:
            return torch.empty(0, 2, device=rois.device, dtype=torch.long), keys.new_empty(0)
        # Geometry controls connectivity; it is never an optimization variable.
        positions = rois[:, 1:3].detach().float()
        sizes = rois[:, 3:5].detach().float().clamp_min(self.cfg["numerical_eps"]).prod(1).sqrt()
        normalized_keys = F.normalize(keys.detach().float(), dim=1)
        for image in torch.unique(rois[:, 0]):
            nodes = torch.nonzero(rois[:, 0] == image).flatten()
            if len(nodes) < 2:
                continue
            distance = torch.cdist(positions[nodes], positions[nodes])
            distance = distance / ((sizes[nodes, None] + sizes[None, nodes]) * 0.5).clamp_min(self.cfg["numerical_eps"])
            distance.fill_diagonal_(float("inf"))
            nearest = torch.topk(distance, min(neighbors, len(nodes) - 1), largest=False).indices
            source = nodes[:, None].expand_as(nearest).reshape(-1)
            destination = nodes[nearest].reshape(-1)
            pairs = torch.stack((torch.minimum(source, destination), torch.maximum(source, destination)), 1)
            edges.append(torch.unique(pairs, dim=0))
        if not edges:
            return torch.empty(0, 2, device=rois.device, dtype=torch.long), keys.new_empty(0)
        edge = torch.cat(edges, 0)
        left, right = edge[:, 0], edge[:, 1]
        scale = 0.5 * (sizes[left] + sizes[right])
        distance2 = ((positions[left] - positions[right]) / scale[:, None]).square().sum(1)
        cosine_distance = (1. - (normalized_keys[left] * normalized_keys[right]).sum(1)).clamp(0., 2.)
        weights = self.cfg["weight_max"] * torch.exp(
            -distance2 / (2. * self.cfg["position_scale"] ** 2)
            -cosine_distance / self.cfg["feature_temperature"])
        return edge, weights

    def forward(self, h0, memory_residual, reliability, rois, keys):
        n = len(h0)
        if h0.ndim != 2 or memory_residual.shape != h0.shape or reliability.shape != (n,):
            raise ValueError("Evidence, residual and reliability dimensions differ")
        if rois.shape != (n, 6) or keys.ndim != 2 or len(keys) != n:
            raise ValueError("RoI/key dimensions do not match evidence")
        if not all(torch.isfinite(t).all() for t in (h0, memory_residual, reliability, rois, keys)):
            raise ValueError("Graph inputs must be finite")
        original_dtype = h0.dtype
        h0, memory_residual = h0.float(), memory_residual.float()
        if not n:
            return h0.to(original_dtype), {"energy_history": h0.new_zeros(1), "edge_count": 0,
                "edges": torch.empty(0, 2, device=h0.device, dtype=torch.long),
                "step_size": h0.new_tensor(0.), "lipschitz_bound": h0.new_tensor(self.cfg["unary_precision"]),
                "gradient_residual": h0.new_tensor(0.), "coefficients_fixed": True}
        # Fixed during this solve; gradients can still flow through the memory tether.
        c = reliability.float().clamp(0., 1.)
        target = h0 + memory_residual
        edges, weights = self._graph(rois, keys)
        left, right = edges[:, 0], edges[:, 1]
        degree = h0.new_zeros(n)
        if len(edges):
            degree = degree.index_add(0, left, weights).index_add(0, right, weights)
        a, eta, lam, delta = (self.cfg[k] for k in ("unary_precision", "eta_memory", "lambda_pairwise", "rho_delta"))
        # The graph is block diagonal by image. Certify each block separately,
        # otherwise a dense unrelated image changes a finite-step prediction.
        image_ids = torch.unique(rois[:, 0])
        node_steps = torch.zeros_like(c)
        image_bounds = []
        for image in image_ids:
            members = rois[:, 0] == image
            image_bound = (a + eta * c[members]).max() + 2. * lam * degree[members].max() / delta
            image_bounds.append(image_bound)
            node_steps[members] = 1. / image_bound
        image_bounds = torch.stack(image_bounds)
        current = h0
        energies = []
        for iteration in range(int(self.cfg["inner_steps"]) + 1):
            difference = current[left] - current[right]
            root = torch.sqrt(difference.square() + delta ** 2)
            energy = (0.5 * a * (current - h0).square().sum()
                      + 0.5 * eta * (c[:, None] * (current - target).square()).sum()
                      + lam * (weights[:, None] * (root - delta)).sum())
            energies.append(energy.detach())
            gradient = a * (current - h0) + eta * c[:, None] * (current - target)
            if len(edges):
                flux = lam * weights[:, None] * difference / root
                gradient = gradient.index_add(0, left, flux).index_add(0, right, -flux)
            if iteration < int(self.cfg["inner_steps"]):
                current = current - node_steps[:, None] * gradient
        diag = {"energy_history": torch.stack(energies), "edge_count": len(edges), "edges": edges,
                "step_size": node_steps.detach().min(), "lipschitz_bound": image_bounds.detach().max(),
                "image_ids": image_ids, "image_lipschitz_bounds": image_bounds.detach(),
                "node_step_sizes": node_steps.detach(),
                "gradient_residual": gradient.detach().norm(), "coefficients_fixed": True,
                "max_weighted_degree": degree.detach().max(), "strong_convexity_floor": a,
                "memory_entry": "single_soft_tether", "iterations": int(self.cfg["inner_steps"])}
        return current.to(original_dtype), diag

In [ ]:
# data
"""Audited 2-D OBB input, exact isotropic preprocessing and rotated evaluation.

Coordinates are image coordinates: x right, y down, angles clockwise in radians.
Every tunable option is read from cfg so it can be exposed in notebook cell two.
"""














def _data_numpy(value, dtype=np.float32):
    if isinstance(value, torch.Tensor):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


def _data_array(value, columns, name):
    array = _data_numpy(value)
    if array.size == 0:
        return np.empty((0, columns), dtype=np.float32)
    if array.ndim != 2 or array.shape[1] != columns or not np.isfinite(array).all():
        raise ValueError(f"{name}: expected finite N x {columns} array")
    return array.copy()


def _data_labels(value, count, classes, name):
    if not isinstance(value, (list, tuple, np.ndarray, torch.Tensor)) or len(value) != count:
        raise ValueError(f"{name}: labels length does not match annotations")
    output = []
    for label in value:
        if isinstance(label, torch.Tensor):
            label = label.item()
        if isinstance(label, str):
            if label not in classes:
                raise ValueError(f"{name}: unknown class {label!r}")
            output.append(classes.index(label))
        elif isinstance(label, (int, np.integer)) and not isinstance(label, (bool, np.bool_)) and 0 <= label < len(classes):
            output.append(int(label))
        else:
            raise ValueError(f"{name}: invalid zero-based label {label!r}")
    return np.asarray(output, dtype=np.int64)


def obb_polygons(boxes):
    """Return N x 8 clockwise corners without changing the input box geometry."""
    boxes = _data_array(boxes, 5, "boxes")
    if not len(boxes):
        return np.empty((0, 8), dtype=np.float32)
    local = np.array([[-.5, -.5], [.5, -.5], [.5, .5], [-.5, .5]], np.float32)
    points = local[None] * boxes[:, None, 2:4]
    cosine, sine = np.cos(boxes[:, 4]), np.sin(boxes[:, 4])
    x = cosine[:, None] * points[:, :, 0] - sine[:, None] * points[:, :, 1]
    y = sine[:, None] * points[:, :, 0] + cosine[:, None] * points[:, :, 1]
    return (np.stack([x, y], axis=-1) + boxes[:, None, :2]).reshape(-1, 8)


def _data_canonical(boxes):
    boxes = boxes.copy()
    swap = boxes[:, 2] < boxes[:, 3]
    boxes[swap, 2:4] = boxes[swap, 2:4][:, ::-1]
    boxes[swap, 4] += np.pi / 2
    boxes[:, 4] = (boxes[:, 4] + np.pi / 2) % np.pi - np.pi / 2
    return boxes


def _data_bounds(points, width, height, tolerance, name):
    points = np.asarray(points).reshape(-1, 2)
    if len(points) and (np.any(points < -tolerance) or np.any(points[:, 0] > width + tolerance) or np.any(points[:, 1] > height + tolerance)):
        raise ValueError(f"{name}: annotation extent is outside image bounds {width} x {height}")


def _data_annotations(record, classes, width, height, cfg):
    name = str(record["id"])
    if ("boxes" in record) == ("polygons" in record):
        raise ValueError(f"{name}: provide exactly one annotation field: boxes or polygons")
    tolerance = float(cfg.get("annotation_bounds_tolerance", 1.0))
    if not math.isfinite(tolerance) or tolerance < 0:
        raise ValueError("annotation_bounds_tolerance must be finite and non-negative")
    if "boxes" in record:
        boxes = _data_array(record["boxes"], 5, name)
        if len(boxes) and (np.any(boxes[:, 2:4] <= 0) or np.any(boxes[:, 4] < -np.pi / 2 - 1e-6) or np.any(boxes[:, 4] >= np.pi / 2 + 1e-6)):
            raise ValueError(f"{name}: boxes need positive extents and le90 angles in radians")
        _data_bounds(obb_polygons(boxes), width, height, tolerance, name)
        boxes = _data_canonical(boxes)
    else:
        polygons = _data_array(record["polygons"], 8, name)
        _data_bounds(polygons, width, height, tolerance, name)
        converted = []
        for polygon in polygons.reshape(-1, 4, 2):
            edges = np.roll(polygon, -1, axis=0) - polygon
            cross = edges[:, 0] * np.roll(edges, -1, axis=0)[:, 1] - edges[:, 1] * np.roll(edges, -1, axis=0)[:, 0]
            if not (np.all(cross > 1e-6) or np.all(cross < -1e-6)):
                raise ValueError(f"{name}: polygon must be strictly convex with ordered, distinct corners")
            (cx, cy), (bw, bh), angle = cv2.minAreaRect(polygon.astype(np.float32))
            if min(bw, bh) <= 0:
                raise ValueError(f"{name}: polygon has zero area")
            converted.append([cx, cy, bw, bh, math.radians(angle)])
        boxes = _data_canonical(np.asarray(converted, dtype=np.float32).reshape(-1, 5))
    if "labels" not in record:
        raise ValueError(f"{name}: labels are missing")
    labels = _data_labels(record["labels"], len(boxes), classes, name)
    difficult = np.asarray(record.get("difficult", [0] * len(boxes)))
    if difficult.shape != (len(boxes),) or not np.isin(difficult, [0, 1, 2]).all():
        raise ValueError(f"{name}: difficult must have one 0/1/2 value per annotation")
    ignored = difficult.astype(bool)
    return boxes[~ignored], labels[~ignored], boxes[ignored], labels[ignored]


def _data_rgb(path):
    try:
        with Image.open(path) as source:
            # Keep raw image coordinates; do not silently rotate labeled EXIF images.
            return np.asarray(source.convert("RGB")).copy()
    except (OSError, ValueError) as error:
        raise ValueError(f"Unable to decode image {path}: {error}") from error


def _data_classes(value):
    if not isinstance(value, (list, tuple)) or not value or any(not isinstance(c, str) or not c.strip() for c in value) or len(set(value)) != len(value):
        raise ValueError("classes must be a non-empty list of distinct class names")
    return list(value)


def _data_dota(root, cfg):
    classes = _data_classes(cfg.get("classes"))
    group_path = Path(cfg.get("groups_csv") or root / "groups.csv")
    groups = {}
    if group_path.is_file():
        with group_path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            if not {"image_id", "group_id"}.issubset(reader.fieldnames or []):
                raise ValueError("groups.csv requires image_id and group_id columns")
            for row in reader:
                key, value = row["image_id"].strip(), row["group_id"].strip()
                if not key or not value or key in groups:
                    raise ValueError("groups.csv has empty or duplicate image_id/group_id entries")
                groups[key] = value
    records = []
    for split in ("train", "val", "test"):
        image_root = root / split / "images"
        if not image_root.exists():
            continue
        images = sorted(p for p in image_root.rglob("*") if p.is_file() and p.suffix.lower() in (".png", ".jpg", ".jpeg"))
        for image in images:
            relative = image.relative_to(image_root)
            key = f"{split}/{relative.with_suffix('').as_posix()}"
            annotation = root / split / "annfiles" / relative.with_suffix(".txt")
            if not annotation.is_file():
                raise ValueError(f"{key}: missing annotation file {annotation}; missing labels are not negatives")
            polygons, labels, difficult = [], [], []
            for line_number, line in enumerate(annotation.read_text(encoding="utf-8-sig").splitlines(), 1):
                if not line.strip() or line.startswith(("imagesource:", "gsd:")):
                    continue
                parts = line.split()
                if len(parts) < 10:
                    raise ValueError(f"{annotation}:{line_number}: expected 8 coordinates, class and difficult")
                try:
                    polygons.append([float(v) for v in parts[:8]])
                    difficult.append(int(parts[-1]))
                except ValueError as error:
                    raise ValueError(f"{annotation}:{line_number}: invalid annotation number") from error
                labels.append(" ".join(parts[8:-1]))
            group = groups.get(key, groups.get(relative.as_posix(), groups.get(relative.with_suffix('').as_posix())))
            records.append(dict(id=key, image=str(image.resolve()), split=split, group_id=group,
                                polygons=polygons, labels=labels, difficult=difficult, reviewed=True))
    return classes, records


def load_data(cfg, required_splits=("train", "val")):
    """Validate a JSON manifest or DOTA split tree; return classes, records, audit.

    A manifest requires classes and records. Each record requires id, image, split,
    group_id, boxes OR polygons, labels and reviewed=true. DOTA empty annotation
    files explicitly represent reviewed negatives; missing files always fail.
    Training requires train/val by default; standalone evaluation can request only
    its selected split, while all supplied records still receive integrity checks.
    """
    if not isinstance(required_splits, (tuple, list)) or any(split not in ("train", "val", "test") for split in required_splits):
        raise ValueError("required_splits must list train, val and/or test")
    manifest_path = cfg.get("manifest_path") or cfg.get("manifest")
    if manifest_path:
        manifest_path = Path(manifest_path).expanduser().resolve()
        payload = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        classes = _data_classes(payload.get("classes"))
        if cfg.get("classes") and list(cfg["classes"]) != classes:
            raise ValueError("cfg classes disagree with manifest classes or their order")
        raw_records = payload.get("records")
        if not isinstance(raw_records, list):
            raise ValueError("manifest records must be a list")
        base = manifest_path.parent
    else:
        if not cfg.get("data_root"):
            raise ValueError("Set manifest or data_root to labeled training data")
        base = Path(cfg["data_root"]).expanduser().resolve()
        classes, raw_records = _data_dota(base, cfg)
    records, ids, group_splits, pixel_splits = [], set(), {}, {}
    pixel_groups = {}
    split_counts = dict(train=0, val=0, test=0)
    automatic_groups, polygon_count, duplicate_within_split = 0, 0, 0
    for raw in raw_records:
        if not isinstance(raw, dict) or not isinstance(raw.get("id"), str) or not raw["id"].strip():
            raise ValueError("Every record needs a non-empty string id")
        record = dict(raw)
        key, split = record["id"], record.get("split")
        if key in ids:
            raise ValueError(f"Duplicate record id {key!r}")
        ids.add(key)
        if split not in split_counts:
            raise ValueError(f"{key}: split must be train, val or test")
        if record.get("reviewed") is not True:
            raise ValueError(f"{key}: reviewed must explicitly be true")
        group = record.get("group_id")
        if group is None or str(group).strip() == "":
            if not cfg.get("allow_image_groups", False):
                raise ValueError(f"{key}: group_id is required; enable allow_image_groups only for an explicit image-level split")
            group = f"image:{key}"
            automatic_groups += 1
        if isinstance(group, (list, dict, bool)):
            raise ValueError(f"{key}: group_id must be a string or scalar patient identifier")
        group = str(group)
        if group in group_splits and group_splits[group] != split:
            raise ValueError(f"{key}: group {group!r} crosses data splits")
        group_splits[group] = split
        if not isinstance(record.get("image"), str) or not record["image"]:
            raise ValueError(f"{key}: image path is missing")
        image = Path(record["image"]).expanduser()
        image = (base / image).resolve() if not image.is_absolute() else image.resolve()
        if image.suffix.lower() not in (".png", ".jpg", ".jpeg"):
            raise ValueError(f"{key}: image must be PNG/JPG/JPEG")
        rgb = _data_rgb(image)
        height, width = rgb.shape[:2]
        digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
        if digest in pixel_splits:
            if pixel_splits[digest] != split:
                raise ValueError(f"{key}: duplicate decoded image pixels cross data splits")
            if pixel_groups[digest] != group:
                raise ValueError(f"{key}: duplicate decoded image pixels belong to different groups; this would bypass same-group reference exclusion")
            duplicate_within_split += 1
        pixel_splits[digest] = split
        pixel_groups[digest] = group
        boxes, labels, ignored_boxes, ignored_labels = _data_annotations(record, classes, width, height, cfg)
        polygon_count += len(record.get("polygons", []))
        record.update(image=str(image), group_id=group, boxes=boxes, labels=labels,
                      ignored_boxes=ignored_boxes, ignored_labels=ignored_labels,
                      width=width, height=height, pixel_sha256=digest)
        records.append(record)
        split_counts[split] += 1
    missing_splits = [split for split in required_splits if not split_counts[split]]
    if missing_splits:
        raise ValueError(f"Required data splits are empty or missing: {', '.join(missing_splits)}")
    audit = dict(split_counts=split_counts, group_count=len(group_splits),
                 grouping="per-image" if automatic_groups == len(records) else ("mixed-explicit-and-image" if automatic_groups else "provided-groups"),
                 automatic_image_groups=automatic_groups, duplicate_pixels_within_split=duplicate_within_split,
                 polygon_to_minimum_area_rectangle_count=polygon_count,
                 negative_images=sum(len(r["boxes"]) == 0 and len(r["ignored_boxes"]) == 0 for r in records),
                 annotation_count=sum(len(r["boxes"]) for r in records),
                 ignored_annotation_count=sum(len(r["ignored_boxes"]) for r in records),
                 pixel_hash="SHA256(decoded RGB shape and pixels)", coordinates="xywha, radians clockwise, le90")
    return classes, records, audit


class OBBDataset(Dataset):
    """Isotropic affine resize and right/bottom pad; no annotation truncation."""
    def __init__(self, records, cfg, training=False):
        self.records = list(records)
        self.training = bool(training)
        self.horizontal_flip_prob = float(cfg.get("train_horizontal_flip_prob", .25))
        self.vertical_flip_prob = float(cfg.get("train_vertical_flip_prob", .25))
        if not 0 <= self.horizontal_flip_prob <= 1 or not 0 <= self.vertical_flip_prob <= 1:
            raise ValueError("Training flip probabilities must be finite numbers between 0 and 1")
        self.size = int(cfg.get("image_size", 1024))
        self.mean = np.asarray(cfg.get("image_mean", [123.675, 116.28, 103.53]), np.float32)
        self.std = np.asarray(cfg.get("image_std", [58.395, 57.12, 57.375]), np.float32)
        if self.size <= 0 or self.size % 32 or self.mean.shape != (3,) or self.std.shape != (3,) or np.any(self.std <= 0) or not np.isfinite(self.mean).all() or not np.isfinite(self.std).all():
            raise ValueError("image_size must be a positive multiple of 32; RGB mean/std must be finite with positive std")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        rgb = _data_rgb(record["image"])
        height, width = rgb.shape[:2]
        boxes = _data_array(record.get("boxes", []), 5, "boxes")
        ignored = _data_array(record.get("ignored_boxes", []), 5, "ignored_boxes")
        horizontal = self.training and np.random.random() < self.horizontal_flip_prob
        vertical = self.training and np.random.random() < self.vertical_flip_prob
        if horizontal:
            rgb = np.flip(rgb, axis=1)
            for annotation in (boxes, ignored):
                # Continuous box coordinates span [0, width], unlike pixel indices.
                annotation[:, 0] = width - annotation[:, 0]
                annotation[:, 4] = -annotation[:, 4]
        if vertical:
            rgb = np.flip(rgb, axis=0)
            for annotation in (boxes, ignored):
                annotation[:, 1] = height - annotation[:, 1]
                annotation[:, 4] = -annotation[:, 4]
        if horizontal or vertical:
            boxes, ignored = _data_canonical(boxes), _data_canonical(ignored)
        scale = self.size / max(height, width)
        rh, rw = min(self.size, math.ceil(height * scale)), min(self.size, math.ceil(width * scale))
        # One scalar transform avoids anisotropic box errors caused by rounding
        # independent cv2.resize dimensions. Pixel centers use half-pixel mapping.
        matrix = np.array([[scale, 0, (scale - 1) * .5], [0, scale, (scale - 1) * .5]], np.float32)
        resized = cv2.warpAffine(rgb.astype(np.float32), matrix, (rw, rh), flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=tuple(float(v) for v in self.mean))
        canvas = np.zeros((self.size, self.size, 3), np.float32)
        canvas[:rh, :rw] = (resized - self.mean) / self.std
        boxes[:, :4] *= scale
        ignored[:, :4] *= scale
        meta = dict(img_shape=(rh, rw, 3), ori_shape=(height, width, 3), pad_shape=(self.size, self.size, 3),
                    scale_factor=np.array([scale] * 4, np.float32), filename=str(record["image"]),
                    ori_filename=Path(record["image"]).name, group_id=record.get("group_id", f"inference:{record['id']}"),
                    sample_id=record["id"], flip=bool(horizontal or vertical),
                    flip_direction="diagonal" if horizontal and vertical else ("horizontal" if horizontal else ("vertical" if vertical else None)),
                    img_norm_cfg=dict(mean=self.mean.copy(), std=self.std.copy(), to_rgb=True))
        return dict(img=torch.from_numpy(canvas.transpose(2, 0, 1).copy()), meta=meta,
                    boxes=torch.from_numpy(boxes), labels=torch.from_numpy(_data_numpy(record.get("labels", []), np.int64)),
                    ignored_boxes=torch.from_numpy(ignored))


def collate_obbs(items):
    if not items:
        raise ValueError("Cannot collate an empty batch")
    return dict(img=torch.stack([item["img"] for item in items]), metas=[item["meta"] for item in items],
                boxes=[item["boxes"] for item in items], labels=[item["labels"] for item in items],
                ignored_boxes=[item["ignored_boxes"] for item in items])


def inference_records(directory, exclude_dirs=()):
    directory = Path(directory).expanduser().resolve()
    if not directory.is_dir():
        raise ValueError(f"Inference directory does not exist: {directory}")
    excluded = tuple(Path(path).expanduser().resolve() for path in exclude_dirs)
    records = []
    for path in sorted(directory.rglob("*")):
        resolved = path.resolve()
        if any(resolved == parent or parent in resolved.parents for parent in excluded):
            continue
        if path.is_file() and path.suffix.lower() in (".png", ".jpg", ".jpeg"):
            key = path.relative_to(directory).as_posix()
            rgb = _data_rgb(path)
            records.append(dict(id=key, image=str(path), group_id=f"inference:{key}",
                                width=rgb.shape[1], height=rgb.shape[0],
                                boxes=np.empty((0, 5), np.float32), labels=np.empty(0, np.int64)))
    if not records:
        raise ValueError("Inference directory has no PNG/JPG/JPEG images")
    return records


def _data_predictions(predictions, record_ids, num_classes):
    if isinstance(predictions, dict):
        entries = [dict(value, id=key) for key, value in predictions.items()]
    else:
        entries = list(predictions)
    output = {}
    for item in entries:
        key = item.get("id", item.get("sample_id"))
        if key not in record_ids or key in output:
            raise ValueError(f"Unknown or duplicate prediction id {key!r}")
        boxes = _data_array(item.get("boxes", []), 5, f"prediction {key}")
        scores = _data_numpy(item.get("scores", []))
        if scores.shape != (len(boxes),) or not np.isfinite(scores).all() or np.any(scores < 0) or np.any(scores > 1):
            raise ValueError(f"{key}: prediction scores must be finite probabilities")
        if len(boxes) and np.any(boxes[:, 2:4] <= 0):
            raise ValueError(f"{key}: prediction box extents must be positive")
        labels = _data_labels(item.get("labels", []), len(boxes), list(range(num_classes)), f"prediction {key}")
        output[key] = dict(boxes=boxes, scores=scores, labels=labels)
    return output


def _data_iou(boxes, targets, device):
    if not len(boxes) or not len(targets):
        return np.zeros((len(boxes), len(targets)), np.float32)
    left = torch.as_tensor(boxes, dtype=torch.float32, device=device).contiguous()
    right = torch.as_tensor(targets, dtype=torch.float32, device=device).contiguous()
    return box_iou_rotated(left, right, clockwise=True).cpu().numpy()


def _data_match(ranked, ground_truth, ignored_truth, threshold, device, overlap_cache=None):
    matched = {key: np.zeros(len(value), dtype=bool) for key, value in ground_truth.items()}
    tp, fp, scores = [], [], []
    # Compute each image's pairwise matrix in one operator call, rather than one
    # CUDA/CPU launch per detection. Keep global score order for matching.
    if overlap_cache is None:
        overlap_cache = {}
    if not overlap_cache:
        image_rows = {}
        for rank, (_, key, box) in enumerate(ranked):
            image_rows.setdefault(key, []).append((rank, box))
        overlaps, ignored_overlaps = {}, {}
        for key, entries in image_rows.items():
            boxes = np.stack([entry[1] for entry in entries])
            regular = _data_iou(boxes, ground_truth[key], device)
            difficult = _data_iou(boxes, ignored_truth[key], device)
            for row, (rank, _) in enumerate(entries):
                overlaps[rank] = regular[row]
                ignored_overlaps[rank] = difficult[row]
        overlap_cache.update(regular=overlaps, difficult=ignored_overlaps)
    overlaps, ignored_overlaps = overlap_cache["regular"], overlap_cache["difficult"]
    for rank, (score, key, _) in enumerate(ranked):
        ious = overlaps[rank]
        available = np.where(~matched[key], ious, -1.)
        best = int(np.argmax(available)) if len(available) else -1
        if best >= 0 and available[best] >= threshold:
            matched[key][best] = True
            tp.append(1.)
            fp.append(0.)
        elif len(ignored_overlaps[rank]) and np.max(ignored_overlaps[rank]) >= threshold:
            continue
        else:
            tp.append(0.)
            fp.append(1.)
        scores.append(score)
    return np.asarray(tp), np.asarray(fp), np.asarray(scores)


def _data_ap101(tp, fp, count):
    if count == 0:
        return None
    if len(tp) == 0:
        return 0.
    cumulative_tp, cumulative_fp = np.cumsum(tp), np.cumsum(fp)
    recall = cumulative_tp / count
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, np.finfo(float).eps)
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    indices = np.searchsorted(recall, np.linspace(0, 1, 101), side="left")
    values = np.zeros(101)
    valid = indices < len(precision)
    values[valid] = precision[indices[valid]]
    return float(values.mean())


def evaluate_obbs(predictions, records, num_classes, cfg):
    """101-point rotated AP at IoUs .50:.05:.95, plus micro precision/recall.

    Predictions and ground truth are in original image coordinates. Classes with
    no non-ignored ground truth are excluded from mAP and reported as None.
    This transparent evaluator is not claimed to reproduce an unavailable
    proprietary baseline evaluator. No COCO maxDet truncation is applied.
    """
    if num_classes <= 0:
        raise ValueError("num_classes must be positive")
    records = list(records)
    keys = [r["id"] for r in records]
    if len(set(keys)) != len(keys):
        raise ValueError("Evaluation records contain duplicate ids")
    normalized = _data_predictions(predictions, set(keys), num_classes)
    device = str(cfg.get("eval_device", "cpu"))
    operating_iou = float(cfg.get("metric_match_iou", cfg.get("eval_iou_threshold", .5)))
    score_cut = float(cfg.get("metric_score_threshold", cfg.get("eval_score_threshold", .5)))
    if not 0 < operating_iou <= 1 or not 0 <= score_cut <= 1:
        raise ValueError("Evaluation IoU/score thresholds are outside valid ranges")
    thresholds = np.round(np.arange(.5, .951, .05), 2)
    ap_matrix = np.full((num_classes, len(thresholds)), np.nan)
    class_counts, operating_counts = {}, []
    for label in range(num_classes):
        truth, ignored = {}, {}
        for record in records:
            boxes = _data_array(record.get("boxes", []), 5, record["id"])
            labels = _data_labels(record.get("labels", []), len(boxes), list(range(num_classes)), record["id"])
            ignored_boxes = _data_array(record.get("ignored_boxes", []), 5, record["id"])
            ignored_labels = _data_labels(record.get("ignored_labels", []), len(ignored_boxes), list(range(num_classes)), record["id"])
            truth[record["id"]] = boxes[labels == label]
            ignored[record["id"]] = ignored_boxes[ignored_labels == label]
        count = sum(len(value) for value in truth.values())
        class_counts[str(label)] = count
        ranked = [(float(score), key, box) for key, prediction in normalized.items()
                  for box, score, cls in zip(prediction["boxes"], prediction["scores"], prediction["labels"]) if cls == label]
        ranked.sort(key=lambda value: -value[0])
        overlap_cache = {}
        for column, threshold in enumerate(thresholds):
            tp, fp, _ = _data_match(ranked, truth, ignored, float(threshold), device, overlap_cache)
            ap = _data_ap101(tp, fp, count)
            if ap is not None:
                ap_matrix[label, column] = ap
        tp, fp, scores = _data_match(ranked, truth, ignored, operating_iou, device, overlap_cache)
        selected = scores >= score_cut
        operating_counts.append((float(tp[selected].sum()), float(fp[selected].sum()), count))
    valid = np.isfinite(ap_matrix[:, 0])
    tp_count, fp_count, gt_count = np.sum(np.asarray(operating_counts), axis=0)
    return {"mAP50:95": float(ap_matrix[valid].mean()) if valid.any() else None,
            "mAP50": float(ap_matrix[valid, 0].mean()) if valid.any() else None,
            "mAP75": float(ap_matrix[valid, 5].mean()) if valid.any() else None,
            "AP50": float(ap_matrix[valid, 0].mean()) if valid.any() else None,
            "AP75": float(ap_matrix[valid, 5].mean()) if valid.any() else None,
            "class_AP": {str(i): float(ap_matrix[i].mean()) if valid[i] else None for i in range(num_classes)},
            "class_AP50": {str(i): float(ap_matrix[i, 0]) if valid[i] else None for i in range(num_classes)},
            "class_gt_counts": class_counts, "precision": float(tp_count / max(tp_count + fp_count, 1)),
            "recall": float(tp_count / max(gt_count, 1)), "true_positives": int(tp_count),
            "false_positives": int(fp_count), "ground_truth_count": int(gt_count),
            "score_threshold": score_cut, "operating_iou": operating_iou,
            "iou_thresholds": thresholds.tolist(), "interpolation": "101-point",
            "max_detections": None, "images": len(records)}


def save_predictions(predictions, records, classes, output_dir, cfg):
    """Write predictions.json and optionally clockwise-box overlays; return JSON path."""
    output_dir = Path(output_dir).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    lookup = {record["id"]: record for record in records}
    normalized = _data_predictions(predictions, set(lookup), len(classes))
    entries = []
    overlay_dir = output_dir / "overlays"
    if cfg.get("save_overlays", True):
        overlay_dir.mkdir(exist_ok=True)
    for key, prediction in normalized.items():
        polygons = obb_polygons(prediction["boxes"])
        entries.append(dict(id=key, boxes=prediction["boxes"].tolist(), scores=prediction["scores"].tolist(),
                            labels=prediction["labels"].tolist(), polygons=polygons.tolist()))
        if cfg.get("save_overlays", True):
            canvas = Image.fromarray(_data_rgb(lookup[key]["image"]))
            draw = ImageDraw.Draw(canvas)
            for polygon, score, label in zip(polygons.reshape(-1, 4, 2), prediction["scores"], prediction["labels"]):
                if score < float(cfg.get("overlay_score_threshold", .5)):
                    continue
                points = [tuple(float(v) for v in point) for point in polygon]
                draw.line(points + points[:1], fill=(0, 255, 0), width=int(cfg.get("overlay_line_width", 2)))
                draw.text(points[0], f"{classes[label]} {score:.3f}", fill=(255, 255, 0))
            filename = hashlib.sha256(key.encode("utf-8")).hexdigest()[:20] + ".png"
            canvas.save(overlay_dir / filename)
            entries[-1]["overlay"] = f"overlays/{filename}"
    target = output_dir / "predictions.json"
    target.write_text(json.dumps(dict(classes=list(classes), coordinates="original-image xywha, radians clockwise, le90",
                                     predictions=entries), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    return str(target)

In [ ]:
# model
"""SRD extensions over the actual OpenMMLab ReDet implementation.

OpenMMLab components are Apache-2.0, Copyright (c) OpenMMLab.
No user baseline checkpoints or original FAA implementation are bundled.
"""






















def safe_multiclass_nms_rotated(multi_bboxes, multi_scores, score_thr, nms,
                                max_num=-1, score_factors=None, return_inds=False):
    """Device-safe classwise rotated NMS; indices refer to flattened candidates."""
    nc = multi_scores.shape[1] - 1
    boxes = multi_bboxes.reshape(-1, nc, 5) if multi_bboxes.shape[1] > 5 else multi_bboxes[:, None].expand(-1, nc, -1)
    scores = multi_scores[:, :nc]
    threshold = nms.get('iou_thr', nms.get('iou_threshold', .1))
    kept_boxes, kept_scores, kept_labels, kept_indices = [], [], [], []
    for c in range(nc):
        valid = (scores[:, c] > score_thr) & torch.isfinite(boxes[:, c]).all(-1)
        valid = valid & (boxes[:, c, 2:4] > 0).all(-1)
        ids = torch.where(valid)[0]
        if ids.numel() == 0:
            continue
        bc = boxes[ids, c].float().contiguous()
        sc = scores[ids, c].float()
        if score_factors is not None:
            sc = sc * score_factors[ids]
        _, keep = nms_rotated(bc, sc.contiguous(), float(threshold))
        kept_boxes.append(bc[keep])
        kept_scores.append(sc[keep])
        kept_labels.append(torch.full_like(keep, c))
        kept_indices.append(ids[keep] * nc + c)
    if not kept_boxes:
        det = multi_bboxes.new_zeros((0, 6))
        labels = torch.empty(0, dtype=torch.long, device=multi_scores.device)
        return (det, labels, labels) if return_inds else (det, labels)
    b, s = torch.cat(kept_boxes), torch.cat(kept_scores)
    labels, ids = torch.cat(kept_labels), torch.cat(kept_indices)
    order = torch.argsort(s, descending=True, stable=True)
    if max_num > 0:
        order = order[:max_num]
    det = torch.cat((b[order], s[order, None]), dim=1)
    return (det, labels[order], ids[order]) if return_inds else (det, labels[order])


def group_code(group):
    return int.from_bytes(hashlib.sha256(str(group).encode('utf-8')).digest()[:8], 'big') & ((1 << 63) - 1)


@BBOX_ASSIGNERS.register_module(force=True)
class RESMaxIoUAssigner(MaxIoUAssigner):
    """Official assignment, including ignored regions when regular GT is empty.

    MMDetection 2.28's empty-GT branch otherwise resets every candidate to
    background after computing the ignore overlaps. Nonempty-GT matching and
    sampling remain the official implementation.
    """
    def assign(self, bboxes, gt_bboxes, gt_bboxes_ignore=None, gt_labels=None):
        result = super().assign(bboxes, gt_bboxes, gt_bboxes_ignore, gt_labels)
        if (gt_bboxes.numel() == 0 and self.ignore_iof_thr > 0
                and gt_bboxes_ignore is not None and gt_bboxes_ignore.numel()
                and bboxes.numel()):
            if self.ignore_wrt_candidates:
                overlap = self.iou_calculator(bboxes, gt_bboxes_ignore, mode='iof').max(dim=1).values
            else:
                overlap = self.iou_calculator(gt_bboxes_ignore, bboxes, mode='iof').max(dim=0).values
            ignored = overlap > self.ignore_iof_thr
            result.gt_inds[ignored] = -1
            result.max_overlaps[ignored] = -1
            if result.labels is not None:
                result.labels[ignored] = -1
        return result


def ignored_obbs_to_hbbs(boxes, version):
    if boxes is None:
        return None
    if boxes.ndim != 2 or boxes.shape[1] != 5:
        raise ValueError('Ignored boxes must use [cx, cy, w, h, angle] OBB coordinates')
    return obb2xyxy(boxes, version)


@ROTATED_HEADS.register_module(force=True)
class RESRoIHead(RoITransRoIHead):
    def __init__(self, res_cfg, **kwargs):
        self.res_cfg = copy.deepcopy(res_cfg)
        cfg = self.res_cfg
        c, d, k = cfg['channels'], cfg['evidence_dim'], cfg['key_dim']
        if any(not isinstance(value, int) or isinstance(value, bool) for value in (c, d, k)) or not (0 < d <= c and 0 < k <= c):
            raise ValueError('QR key/evidence dimensions must be positive integers no larger than channels')
        super().__init__(**kwargs)
        if self.num_stages != 2:
            raise ValueError('The extension requires exactly two RoI stages and the fixed 2FC head')
        head = self.bbox_head[1]
        if (head.num_shared_convs != 0 or head.num_shared_fcs != 2
                or any(getattr(head, name) != 0 for name in ('num_cls_convs', 'num_cls_fcs', 'num_reg_convs', 'num_reg_fcs'))
                or head.with_avg_pool or not head.with_cls or not head.with_reg):
            raise ValueError('Stage two must use the fixed shared 2FC head without average pooling or separate branches')
        if head.in_channels != c or self.bbox_roi_extractor[1].out_channels != c:
            raise ValueError('Reasoning channels must match the official stage-two RoI extractor and head')
        self.spectral = SpectralRoIAdapter(c, cfg['spectral'])
        self.memory = BayesianMemory(k, d, cfg['memory'])
        self.reasoner = SmoothTVReasoner(cfg['mrf'])
        self.evidence_down = nn.Linear(self.bbox_head[1].fc_out_channels, d)
        self.evidence_up = nn.Linear(d, self.bbox_head[1].fc_out_channels, bias=False)
        self.memory_out = nn.Linear(d, d, bias=False)
        self.evidence_scale = nn.Parameter(torch.full((self.bbox_head[1].fc_out_channels,), cfg['evidence_layer_scale']))
        generator = torch.Generator().manual_seed(cfg['projection_seed'])
        basis, _ = torch.linalg.qr(torch.randn(c, max(d, k), generator=generator), mode='reduced')
        self.register_buffer('key_projection', basis[:, :k].clone())
        self.register_buffer('value_projection', basis[:, :d].clone())
        self.enhancements_enabled = False
        self.current_groups = None
        self.last_diagnostics = {}

    def raw_key_value(self, roi_features):
        pooled = roi_features.float().mean(dim=(-2, -1))
        return pooled @ self.key_projection, pooled @ self.value_projection

    def shared_features(self, feats):
        head = self.bbox_head[1]
        x = feats.flatten(1)
        for fc in head.shared_fcs:
            x = head.relu(fc(x))
        return x

    def _bbox_forward(self, stage, x, rois):
        if stage == 0 or not self.enhancements_enabled:
            return super()._bbox_forward(stage, x, rois)
        extractor = self.bbox_roi_extractor[1]
        raw = extractor(x[:extractor.num_inputs], rois)
        head = self.bbox_head[1]
        if rois.shape[0] == 0:
            cls, reg = head(raw)
            self.last_diagnostics = {'action': 'empty_proposals', 'rois': 0}
            return dict(cls_score=cls, bbox_pred=reg, bbox_feats=raw)
        raw_keys, _ = self.raw_key_value(raw.detach())
        h_shared = self.shared_features(raw)
        class_prob = head.fc_cls(h_shared).float().softmax(-1).detach()
        if self.res_cfg['spectral_enabled']:
            adapted, spectral_diag = self.spectral(raw, rois.detach(), class_prob=class_prob)
            selected = spectral_diag['selected_indices']
            if selected.numel():
                h_shared = h_shared.clone()
                h_shared[selected] = self.shared_features(adapted[selected])
        else:
            spectral_diag = {'fft_rois': 0, 'action': 'disabled'}
        with torch.autocast(device_type=raw.device.type, enabled=False):
            h0 = self.evidence_down(h_shared.float())
            if self.res_cfg['memory_enabled']:
                query_groups = None
                if self.training and self.current_groups is not None:
                    all_groups = torch.tensor([group_code(g) for g in self.current_groups], device=rois.device, dtype=torch.long)
                    query_groups = all_groups[rois[:, 0].long()]
                r, reliability, memory_diag = self.memory(raw_keys.float(), query_groups, query_image_ids=rois[:, 0].long())
                r = self.memory_out(r.float())
            else:
                r, reliability = torch.zeros_like(h0), h0.new_zeros(h0.shape[0])
                memory_diag = {'action': 'disabled'}
            if self.res_cfg['mrf_enabled'] or self.res_cfg['memory_enabled']:
                h, graph_diag = self.reasoner(h0, r, reliability, rois.float(), raw_keys.float())
                h_shared = h_shared.float() + self.evidence_scale * self.evidence_up(h - h0)
            else:
                graph_diag = {'action': 'disabled'}
        cls, reg = head.fc_cls(h_shared), head.fc_reg(h_shared)
        self.last_diagnostics = {'action': 'evidence_refinement', 'rois': int(rois.shape[0]),
                                 'spectral': spectral_diag, 'memory': memory_diag, 'mrf': graph_diag}
        return dict(cls_score=cls, bbox_pred=reg, bbox_feats=raw)

    def forward_train(self, x, img_metas, proposal_list, gt_bboxes, gt_labels,
                      gt_bboxes_ignore=None, gt_masks=None):
        """Official cascade flow with ignore boxes in each stage's coordinates.

        Adapted from MMRotate 0.3.4 RoITransRoIHead.forward_train (Apache-2.0).
        Assignment, sampling, targets, losses and detached refinement are kept.
        """
        self.current_groups = [m['group_id'] for m in img_metas]
        ignored = gt_bboxes_ignore if gt_bboxes_ignore is not None else [None] * len(img_metas)
        if len(ignored) != len(img_metas):
            raise ValueError('Ignored box list must contain one entry per image')
        losses = {}
        for stage in range(self.num_stages):
            self.current_stage = stage
            train_cfg = self.train_cfg[stage]
            sampling_results = []
            for image_index in range(len(img_metas)):
                stage_gt = obb2xyxy(gt_bboxes[image_index], self.version) if stage == 0 else gt_bboxes[image_index]
                stage_ignore = ignored_obbs_to_hbbs(ignored[image_index], self.version) if stage == 0 else ignored[image_index]
                assigned = self.bbox_assigner[stage].assign(
                    proposal_list[image_index], stage_gt, stage_ignore, gt_labels[image_index])
                sampled = self.bbox_sampler[stage].sample(
                    assigned, proposal_list[image_index], stage_gt, gt_labels[image_index],
                    feats=[level[image_index][None] for level in x])
                sampled.pos_gt_bboxes = gt_bboxes[image_index][sampled.pos_assigned_gt_inds]
                sampling_results.append(sampled)
            bbox_results = self._bbox_forward_train(
                stage, x, sampling_results, gt_bboxes, gt_labels, train_cfg)
            for name, value in bbox_results['loss_bbox'].items():
                losses[f's{stage}.{name}'] = value * self.stage_loss_weights[stage] if 'loss' in name else value
            if stage < self.num_stages - 1:
                pos_is_gts = [sample.pos_is_gt for sample in sampling_results]
                roi_labels = bbox_results['bbox_targets'][0]
                with torch.no_grad():
                    cls_score = bbox_results['cls_score']
                    if self.bbox_head[stage].custom_activation:
                        cls_score = self.bbox_head[stage].loss_cls.get_activation(cls_score)
                    roi_labels = torch.where(
                        roi_labels == self.bbox_head[stage].num_classes,
                        cls_score[:, :-1].argmax(1), roi_labels)
                    proposal_list = self.bbox_head[stage].refine_bboxes(
                        bbox_results['rois'], roi_labels, bbox_results['bbox_pred'], pos_is_gts, img_metas)
        return losses

    def simple_test(self, x, proposal_list, img_metas, rescale=False):
        self.current_groups = None
        return super().simple_test(x, proposal_list, img_metas, rescale=rescale)


@ROTATED_DETECTORS.register_module(force=True)
class RESReDet(ReDet):
    """Official ReDet with a stage-aware freeze contract."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.res_stage = 'A'

    def forward_train(self, img, img_metas, gt_bboxes, gt_labels,
                      gt_bboxes_ignore=None, gt_masks=None, proposals=None, **kwargs):
        """Official two-stage flow; RPN's horizontal assigner receives HBB ignores."""
        features = self.extract_feat(img)
        losses = {}
        if self.with_rpn:
            ignored_hbbs = None if gt_bboxes_ignore is None else [
                ignored_obbs_to_hbbs(boxes, self.rpn_head.version) for boxes in gt_bboxes_ignore]
            proposal_cfg = self.train_cfg.get('rpn_proposal', self.test_cfg.rpn)
            rpn_losses, proposal_list = self.rpn_head.forward_train(
                features, img_metas, gt_bboxes, gt_labels=None,
                gt_bboxes_ignore=ignored_hbbs, proposal_cfg=proposal_cfg, **kwargs)
            losses.update(rpn_losses)
        else:
            proposal_list = proposals
        losses.update(self.roi_head.forward_train(
            features, img_metas, proposal_list, gt_bboxes, gt_labels,
            gt_bboxes_ignore, gt_masks, **kwargs))
        return losses

    def train(self, mode=True):
        super().train(mode)
        if self.res_stage == 'B':
            self.backbone.eval()
            self.neck.eval()
            self.rpn_head.eval()
            self.roi_head.bbox_head[0].eval()
        return self


def configure_stage(model, stage, cfg):
    if stage not in ('A', 'B'):
        raise ValueError('stage must be A or B')
    model.res_stage = stage
    for p in model.parameters():
        p.requires_grad_(True)
    model.roi_head.enhancements_enabled = stage == 'B'
    if stage == 'A':
        for module in [model.roi_head.spectral, model.roi_head.memory, model.roi_head.reasoner,
                       model.roi_head.evidence_down, model.roi_head.evidence_up, model.roi_head.memory_out]:
            for p in module.parameters():
                p.requires_grad_(False)
        model.roi_head.evidence_scale.requires_grad_(False)
    else:
        for module in [model.backbone, model.neck, model.rpn_head, model.roi_head.bbox_head[0]]:
            for p in module.parameters():
                p.requires_grad_(False)
    model.train(model.training)
    return model


def build_resroit(cfg, classes, initialize=True):
    # e2cnn 0.2.3 caches mutable nn.Modules, whose sampled_basis buffers follow
    # an existing model onto CUDA. A new CPU model must not share those modules.
    # Releasing cache references preserves every live model's owned bases.
    e2cnn_basis_cache._stored_filters.clear()
    rotated_bbox_head_module.multiclass_nms_rotated = safe_multiclass_nms_rotated
    for name in ('mmcv', 'mmrotate', 'mmdet'):
        logging.getLogger(name).setLevel(logging.ERROR)
    mc = copy.deepcopy(cfg['model'])
    mc['type'] = 'RESReDet'
    mc['roi_head']['type'] = 'RESRoIHead'
    mechanism = copy.deepcopy(cfg['reasoning'])
    if not mechanism['mrf_enabled']:
        mechanism['mrf']['lambda_pairwise'] = 0.0
    mechanism['memory']['num_classes'] = len(classes)
    mc['roi_head']['res_cfg'] = mechanism
    ignore_threshold = float(cfg.get('ignore_iof_threshold', .5))
    if not 0 < ignore_threshold <= 1:
        raise ValueError('ignore_iof_threshold must lie in (0, 1]')
    for stage in [mc['train_cfg']['rpn'], *mc['train_cfg']['rcnn']]:
        if stage['assigner']['type'] not in ('MaxIoUAssigner', 'RESMaxIoUAssigner'):
            raise ValueError('The fixed ReDet cascade requires the official MaxIoU assignment contract')
        stage['assigner'].update(type='RESMaxIoUAssigner', ignore_iof_thr=ignore_threshold)
    score_threshold, max_detections = float(cfg['score_threshold']), int(cfg['max_detections'])
    if not 0 <= score_threshold <= 1 or max_detections < 1:
        raise ValueError('score_threshold must lie in [0,1] and max_detections must be positive')
    mc['test_cfg']['rcnn'].update(score_thr=score_threshold, max_per_img=max_detections)
    for h in mc['roi_head']['bbox_head']:
        h['num_classes'] = len(classes)
    pretrained = cfg['pretrained_backbone']
    mc['backbone']['pretrained'] = pretrained or None
    mc['backbone']['frozen_stages'] = cfg['pretrained_frozen_stages'] if pretrained else -1
    model = build_detector(ConfigDict(mc))
    if initialize:
        model.init_weights()
    model.CLASSES = tuple(classes)
    return configure_stage(model, 'A', cfg)

In [ ]:
# pipeline
"""Training, frozen train-only memory, safe checkpoints and image-only inference."""





















def plain(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): plain(v) for k,v in value.items()}
    if isinstance(value, (tuple,list)):
        return [plain(v) for v in value]
    return value


def write_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(plain(value),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')


def dataset_signature(records, classes):
    keys=('id','split','group_id','pixel_sha256','boxes','labels','ignored_boxes','ignored_labels')
    payload=dict(classes=classes,records=[{k:plain(r.get(k)) for k in keys} for r in sorted(records,key=lambda r:r['id'])])
    return hashlib.sha256(json.dumps(payload,sort_keys=True,allow_nan=False).encode()).hexdigest()


def capture_rng():
    state=np.random.get_state()
    return dict(python=random.getstate(),numpy=[state[0],state[1].tolist(),int(state[2]),int(state[3]),float(state[4])],
                torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state['python'])
    n=state['numpy']; np.random.set_state((n[0],np.asarray(n[1],dtype=np.uint32),n[2],n[3],n[4]))
    torch.set_rng_state(state['torch'].cpu())
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda']) != torch.cuda.device_count():
            raise ValueError('Resume requires the same CUDA device count for RNG replay')
        torch.cuda.set_rng_state_all([v.cpu() for v in state['cuda']])


def validate_config(cfg):
    if cfg['mode'] not in ('train','resume','infer','evaluate'):
        raise ValueError('mode must be train / resume / infer / evaluate')
    for key in ('image_size','batch_size','accumulation_steps','num_threads','stage_a_epochs','stage_b_epochs','validation_interval','checkpoint_every','bank_max_instances_per_chunk'):
        if not isinstance(cfg[key],int) or isinstance(cfg[key],bool) or cfg[key]<=0:
            raise ValueError(f'{key} must be a positive integer')
    if cfg['image_size']%32:
        raise ValueError('image_size must be divisible by 32')
    if cfg['num_workers'] != 0:
        raise ValueError('This portable notebook uses num_workers=0; notebook-defined Dataset classes cannot be spawned safely on Windows')
    if cfg['optimizer'] not in ('SGD','AdamW'):
        raise ValueError('optimizer must be SGD or AdamW')
    for key in ('learning_rate_a','learning_rate_b','gradient_clip'):
        if not math.isfinite(cfg[key]) or cfg[key]<=0:
            raise ValueError(f'{key} must be finite and positive')
    if not 0<=cfg['score_threshold']<=1 or cfg['max_detections']<=0:
        raise ValueError('invalid score_threshold or max_detections')
    if cfg['amp']:
        raise ValueError('amp=True is not supported by the pinned C8/RiRoI engine. Keep amp=False for verified FP32 execution.')
    if cfg['device']=='auto':
        cfg['device']='cuda' if torch.cuda.is_available() else 'cpu'
    if str(cfg['device']).startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable')
    if not str(mmcv.__version__).startswith('1.7.') or mmdet.__version__!='2.28.2' or mmrotate.__version__!='0.3.4':
        raise RuntimeError('Use mmcv-full 1.7.2, mmdet 2.28.2 and mmrotate 0.3.4; newer APIs differ')
    if not str(cfg['device']).startswith('cuda'):
        raise RuntimeError('Full RiRoIAlignRotated detection requires the verified CUDA environment. CPU is supported only for isolated math/data tests.')
    return cfg


def setup_seed(cfg):
    random.seed(cfg['seed']); np.random.seed(cfg['seed']); torch.manual_seed(cfg['seed'])
    torch.cuda.manual_seed_all(cfg['seed']); torch.set_num_threads(cfg['num_threads'])
    torch.backends.cudnn.benchmark=not cfg['deterministic']
    torch.backends.cudnn.deterministic=cfg['deterministic']


def file_sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()


def prepare_pretrained(cfg):
    if cfg['pretrained_backbone']:
        path=Path(cfg['pretrained_backbone']).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f'pretrained_backbone not found: {path}')
        if file_sha256(path)!=cfg['pretrained_sha256']:
            raise ValueError('Local pretrained SHA256 differs from the specified official checkpoint')
        cfg['pretrained_backbone']=str(path)
        return
    if not cfg['download_pretrained']:
        return
    path=Path(cfg['output_dir'])/'cache'/'re_resnet50_c8_batch256-25b16846.pth'
    path.parent.mkdir(parents=True,exist_ok=True)
    expected=cfg['pretrained_sha256']
    if not path.exists() or file_sha256(path)!=expected:
        temporary=path.with_suffix('.download')
        print('Downloading verified official C8 ImageNet initialization...')
        urllib.request.urlretrieve(cfg['pretrained_url'],temporary)
        if file_sha256(temporary)!=expected:
            raise RuntimeError('Downloaded pretrained SHA256 mismatch; file was not loaded')
        os.replace(temporary,path)
    cfg['pretrained_backbone']=str(path.resolve())


def keyspace_signature(model):
    digest=hashlib.sha256()
    for name,tensor in model.state_dict().items():
        if name.startswith(('backbone.','neck.')) or name in ('roi_head.key_projection','roi_head.value_projection'):
            if name.endswith(('.filter','.expanded_bias')):
                continue
            digest.update(name.encode()); digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def save_checkpoint(path, model, cfg, classes, stage, epoch, best_metric, bad_epochs,
                    signature, optimizer, scaler, history, bank_keyspace):
    # e2cnn expanded filters are derived caches, not independent model state.
    weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items() if not k.endswith(('.filter','.expanded_bias'))}
    payload=dict(format='RES-RoIT-BM-1',run_id=cfg.get('_run_id','unspecified'),weights=weights,cfg=copy.deepcopy(cfg),classes=list(classes),stage=stage,
                 epoch=int(epoch),best_metric=float(best_metric),bad_epochs=int(bad_epochs),dataset_signature=signature,
                 optimizer=optimizer.state_dict() if optimizer else None,scaler=scaler.state_dict() if scaler else None,
                 history=copy.deepcopy(history),rng=capture_rng(),bank_keyspace=bank_keyspace,
                 versions=dict(torch=str(torch.__version__),mmcv=mmcv.__version__,mmdet=mmdet.__version__,mmrotate=mmrotate.__version__),
                 checkpoint_contract='completed epoch; no partial accumulation state')
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp'); torch.save(payload,tmp); os.replace(tmp,path)


def load_checkpoint(path, device):
    payload=torch.load(str(path),map_location='cpu',weights_only=True)
    if payload.get('format')!='RES-RoIT-BM-1':
        raise ValueError('Not a compatible SRD checkpoint')
    # initialize=False never reads the original pretrained file. Preserve the
    # frozen-stage setting when resuming an ImageNet-initialized Stage A.
    cfg=copy.deepcopy(payload['cfg'])
    model=build_resroit(cfg,payload['classes'],initialize=False)
    result=model.load_state_dict(payload['weights'],strict=False)
    missing=[k for k in result.missing_keys if not k.endswith(('.filter','.expanded_bias'))]
    if missing or result.unexpected_keys:
        raise RuntimeError(f'Checkpoint architecture mismatch: missing={missing}, extra={result.unexpected_keys}')
    # Go through train before eval to invalidate any stale expanded C8 filters.
    model.train(); model.to(device); configure_stage(model,payload['stage'],payload['cfg']); model.eval()
    if payload['stage']=='B' and payload['bank_keyspace'] and keyspace_signature(model)!=payload['bank_keyspace']:
        raise RuntimeError('Frozen reference-bank key space differs from checkpoint backbone/FPN')
    return model,payload


def compact_diagnostics(value):
    if isinstance(value,torch.Tensor):
        v=value.detach()
        if v.numel()<=12:
            return plain(v)
        return dict(shape=list(v.shape),min=float(v.min()),max=float(v.max()),mean=float(v.float().mean()))
    if isinstance(value,dict):
        return {k:compact_diagnostics(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):
        return [compact_diagnostics(v) for v in value]
    return plain(value)


@torch.no_grad()
def predict_records(model, records, cfg):
    model.eval(); device=next(model.parameters()).device
    predictions={}; logs=[]; timings=[]
    dataset=OBBDataset(records,cfg)
    for item,record in zip(dataset,records):
        if device.type=='cuda': torch.cuda.synchronize(device)
        start=time.perf_counter()
        result=model.simple_test(item['img'][None].to(device),[item['meta']],rescale=True)[0]
        if device.type=='cuda': torch.cuda.synchronize(device)
        timings.append(time.perf_counter()-start)
        boxes=[]; scores=[]; labels=[]
        for label,arr in enumerate(result):
            arr=np.asarray(arr,dtype=np.float32).reshape(-1,6)
            boxes.extend(arr[:,:5].tolist()); scores.extend(arr[:,5].tolist()); labels.extend([label]*len(arr))
        predictions[record['id']]=dict(boxes=boxes,scores=scores,labels=labels)
        logs.append(dict(id=record['id'],diagnostics=compact_diagnostics(model.roi_head.last_diagnostics)))
    timing=dict(images=len(timings),seconds=sum(timings),seconds_per_image=float(np.mean(timings)) if timings else None,
                scope='device transfer + model + NMS; excludes disk image decode, metric and overlay; includes first-call warmup')
    return predictions,logs,timing


@torch.no_grad()
def build_reference_bank(model, train_records, cfg):
    if any(r['split']!='train' for r in train_records):
        raise ValueError('Only train annotations may enter the memory bank')
    model.eval(); device=next(model.parameters()).device
    keys=[]; values=[]; labels=[]; groups=[]
    extractor=model.roi_head.bbox_roi_extractor[1]
    for item in OBBDataset(train_records,cfg):
        boxes=item['boxes'].to(device)
        if not len(boxes): continue
        features=model.extract_feat(item['img'][None].to(device))
        for start in range(0,len(boxes),cfg['bank_max_instances_per_chunk']):
            selected=boxes[start:start+cfg['bank_max_instances_per_chunk']]
            rois=torch.cat((selected.new_zeros(len(selected),1),selected),1)
            raw=extractor(features[:extractor.num_inputs],rois)
            k,v=model.roi_head.raw_key_value(raw)
            keys.append(k.cpu()); values.append(v.cpu())
            labels.append(item['labels'][start:start+len(selected)])
            groups.append(torch.full((len(selected),),group_code(item['meta']['group_id']),dtype=torch.long))
    if not keys:
        raise ValueError('Training data has no positive annotated boxes; cannot train this detector or build a bank')
    model.roi_head.memory.set_bank(torch.cat(keys).to(device),torch.cat(values).to(device),torch.cat(labels).to(device),torch.cat(groups).to(device),source_split='train')
    return keyspace_signature(model)


def make_optimizer(model,cfg,stage):
    params=[p for p in model.parameters() if p.requires_grad]
    lr=cfg['learning_rate_a' if stage=='A' else 'learning_rate_b']
    if cfg['optimizer']=='SGD':
        return torch.optim.SGD(params,lr=lr,momentum=cfg['momentum'],weight_decay=cfg['weight_decay'])
    return torch.optim.AdamW(params,lr=lr,weight_decay=cfg['weight_decay'],betas=tuple(cfg['adam_betas']),eps=cfg['adam_eps'])


def stage_learning_rate(cfg,stage,epoch,update,updates_per_epoch,total_epochs):
    base=cfg['learning_rate_a' if stage=='A' else 'learning_rate_b']
    step=epoch*updates_per_epoch+update
    warm=min(1.,step/max(1,cfg['warmup_steps'])) if cfg['warmup_steps'] else 1.
    warm=cfg['warmup_ratio']+(1-cfg['warmup_ratio'])*warm
    decay=sum(epoch>=max(1,int(total_epochs*m)) for m in cfg['lr_decay_milestones'])
    return base*warm*cfg['lr_decay_gamma']**decay


def verify_checkpoint_pair(best_path,resume):
    selected=torch.load(best_path,map_location='cpu',weights_only=True)
    for key in ('run_id','dataset_signature','classes','stage','bank_keyspace','best_metric'):
        if selected.get(key)!=resume.get(key):
            raise ValueError(f'Best/last checkpoint pair differs in {key}; copy the matching pair from this experiment')
    if selected['epoch']>resume['epoch']:
        raise ValueError('The best checkpoint comes from a later epoch than the resume checkpoint')


def train_stage(model,stage,train_records,val_records,classes,cfg,signature,bank_keyspace,resume=None):
    configure_stage(model,stage,cfg)
    optimizer=make_optimizer(model,cfg,stage)
    epochs=cfg['stage_a_epochs' if stage=='A' else 'stage_b_epochs']
    first=0; best=-1.; bad=0; history=[]
    output=Path(cfg['output_dir']); best_path=output/f'stage_{stage}_best.pth'
    if resume:
        first=resume['epoch']; best=resume['best_metric']; bad=resume['bad_epochs']; history=resume['history']
        optimizer.load_state_dict(resume['optimizer']); restore_rng(resume['rng'])
        # Relocated resumes require a copied best checkpoint, preserving model selection.
        if first and not best_path.is_file():
            raise FileNotFoundError(f'Resume needs the corresponding best checkpoint beside last: {best_path}')
        if first:
            verify_checkpoint_pair(best_path,resume)
    loader=DataLoader(OBBDataset(train_records,cfg,training=True),batch_size=cfg['batch_size'],shuffle=True,
                      num_workers=0,collate_fn=collate_obbs,pin_memory=True,drop_last=False)
    device=next(model.parameters()).device
    updates=math.ceil(len(loader)/cfg['accumulation_steps'])
    for epoch in range(first,epochs):
        if bad>=cfg['early_stopping_patience'] and epoch>=cfg['early_stopping_start_epoch']:
            break
        model.train(); optimizer.zero_grad(set_to_none=True)
        loss_totals={}; start=time.perf_counter()
        for index,batch in enumerate(loader):
            window_start=(index//cfg['accumulation_steps'])*cfg['accumulation_steps']
            window_images=min(cfg['accumulation_steps']*cfg['batch_size'],len(train_records)-window_start*cfg['batch_size'])
            batch_weight=len(batch['metas'])/window_images
            lr=stage_learning_rate(cfg,stage,epoch,index//cfg['accumulation_steps'],updates,epochs)
            for group in optimizer.param_groups: group['lr']=lr
            losses=model.forward_train(batch['img'].to(device,non_blocking=True),batch['metas'],
                                       [v.to(device) for v in batch['boxes']],[v.to(device) for v in batch['labels']],
                                       gt_bboxes_ignore=[v.to(device) for v in batch['ignored_boxes']])
            reduced={k:(sum(vv.mean() for vv in v) if isinstance(v,list) else v.mean()) for k,v in losses.items()}
            total=sum(v for k,v in reduced.items() if 'loss' in k)
            if not torch.isfinite(total):
                raise FloatingPointError(f'Nonfinite loss at stage {stage}, epoch {epoch+1}, batch {index}')
            (total*batch_weight).backward()
            for key,value in reduced.items(): loss_totals[key]=loss_totals.get(key,0.)+float(value.detach())
            if (index+1)%cfg['accumulation_steps']==0 or index+1==len(loader):
                norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],cfg['gradient_clip'],error_if_nonfinite=True)
                optimizer.step(); optimizer.zero_grad(set_to_none=True)
        metrics=None; improved=False
        # Always validate the first/final epoch to guarantee a usable selected checkpoint.
        if (epoch+1)%cfg['validation_interval']==0 or epoch==0 or epoch+1==epochs:
            predictions,_,_=predict_records(model,val_records,cfg)
            metrics=evaluate_obbs(predictions,val_records,len(classes),cfg)
            score=metrics.get(cfg['monitor_metric'])
            if score is None or not math.isfinite(score):
                raise ValueError('Validation monitor is undefined; validation must contain positive ground truth and a supported metric')
            if score>best+cfg['early_stopping_min_delta']:
                best=float(score); bad=0; improved=True
            elif epoch+1>=cfg['early_stopping_start_epoch']:
                bad+=1
        row=dict(stage=stage,epoch=epoch+1,losses={k:v/len(loader) for k,v in loss_totals.items()},
                 validation=metrics,learning_rate=lr,seconds=time.perf_counter()-start,
                 gradient_norm=float(norm),diagnostics=compact_diagnostics(model.roi_head.last_diagnostics))
        history.append(row); write_json(output/f'history_{stage}.json',history)
        if improved:
            save_checkpoint(best_path,model,cfg,classes,stage,epoch+1,best,bad,signature,optimizer,None,history,bank_keyspace)
        if (epoch+1)%cfg['checkpoint_every']==0 or epoch+1==epochs or bad>=cfg['early_stopping_patience']:
            save_checkpoint(output/f'stage_{stage}_last.pth',model,cfg,classes,stage,epoch+1,best,bad,signature,optimizer,None,history,bank_keyspace)
        print(f'Stage {stage} epoch {epoch+1}/{epochs}: loss={sum(v for k,v in row["losses"].items() if "loss" in k):.4f}; val {cfg["monitor_metric"]}={None if metrics is None else metrics[cfg["monitor_metric"]]}; best={best:.6f}')
    if not best_path.is_file():
        raise RuntimeError('No validated checkpoint was produced')
    return best_path


def run_model(user_cfg):
    cfg=copy.deepcopy(user_cfg)
    output=Path(cfg['output_dir']).expanduser().resolve(); cfg['output_dir']=str(output)
    output.mkdir(parents=True,exist_ok=True)
    validate_config(cfg); setup_seed(cfg)
    started=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    environment=dict(python=platform.python_version(),torch=str(torch.__version__),cuda=torch.version.cuda,
                     device=torch.cuda.get_device_name(torch.device(cfg['device'])),mmcv=mmcv.__version__,mmdet=mmdet.__version__,mmrotate=mmrotate.__version__,
                     determinism='seeded; custom CUDA kernels may remain nondeterministic')
    mode=cfg['mode']; checkpoint=cfg['checkpoint_path']
    if mode in ('infer','evaluate'):
        if not checkpoint: raise ValueError('Set checkpoint_path to your trained SRD checkpoint')
        model,payload=load_checkpoint(checkpoint,cfg['device']); classes=payload['classes']
        # Preprocessing/model controls are checkpoint-bound. Explicit overrides are logged.
        runtime=copy.deepcopy(payload['cfg'])
        for k in ('device','output_dir','save_overlays','overlay_score_threshold','overlay_line_width','eval_device','metric_score_threshold','metric_match_iou'):
            runtime[k]=cfg[k]
        for k,v in cfg['inference_overrides'].items():
            if k not in ('score_threshold','max_detections'):
                raise ValueError(f'Unsupported inference override: {k}')
            if k=='score_threshold' and not (isinstance(v,(int,float)) and math.isfinite(v) and 0<=v<=1):
                raise ValueError('Inference score_threshold must lie in [0,1]')
            if k=='max_detections' and not (isinstance(v,int) and not isinstance(v,bool) and v>0):
                raise ValueError('Inference max_detections must be a positive integer')
            runtime[k]=v
        model.roi_head.test_cfg.score_thr=runtime['score_threshold']; model.roi_head.test_cfg.max_per_img=runtime['max_detections']
        if mode=='infer':
            records=inference_records(cfg['inference_dir'],exclude_dirs=(output,)); audit=None
        else:
            declared,all_records,audit=load_data(cfg,required_splits=(cfg['evaluation_split'],))
            if declared!=classes: raise ValueError('Evaluation class order differs from checkpoint')
            records=[r for r in all_records if r['split']==cfg['evaluation_split']]
            if not records: raise ValueError('Requested evaluation split is empty')
        predictions,logs,timing=predict_records(model,records,runtime)
        prediction_file=save_predictions(predictions,records,classes,output,runtime)
        metrics=evaluate_obbs(predictions,records,len(classes),runtime) if mode=='evaluate' else None
        write_json(output/'agent_log.json',logs)
        result=dict(mode=mode,classes=classes,checkpoint=str(Path(checkpoint).resolve()),prediction_file=prediction_file,
                    metrics=metrics,timing=timing,environment=environment,data_audit=audit,inference_overrides=cfg['inference_overrides'])
    else:
        classes,records,audit=load_data(cfg)
        # The fingerprint is order invariant; canonicalize iteration as well so
        # relocated/reordered manifests retain the same seeded resume trajectory.
        records=sorted(records,key=lambda record:record['id'])
        signature=dataset_signature(records,classes)
        train_records=[r for r in records if r['split']=='train']; val_records=[r for r in records if r['split']=='val']
        if not any(len(r['boxes']) for r in train_records) or not any(len(r['boxes']) for r in val_records):
            raise ValueError('Training and validation splits each require positive annotated objects')
        write_json(output/'data_audit.json',audit)
        resume=None; bank_keyspace=None
        if mode=='resume':
            if not checkpoint: raise ValueError('Resume requires checkpoint_path')
            model,resume=load_checkpoint(checkpoint,cfg['device'])
            if signature!=resume['dataset_signature'] or classes!=resume['classes']:
                raise ValueError('Resume dataset/class fingerprint changed; start a new run instead')
            # Resume preserves the experiment; only paths/runtime rendering may move.
            movable={'mode','manifest_path','data_root','inference_dir','output_dir','checkpoint_path','classes','groups_csv',
                     'download_pretrained','pretrained_url','pretrained_sha256','pretrained_backbone','device','num_threads',
                     'evaluate_test','save_overlays','evaluation_split','inference_overrides','overlay_score_threshold','overlay_line_width','eval_device'}
            for key in set(cfg)-movable:
                if cfg[key]!=resume['cfg'][key]: raise ValueError(f'Resume setting changed: {key}')
            cfg['pretrained_backbone']=resume['cfg']['pretrained_backbone']
            cfg['_run_id']=resume['run_id']
            bank_keyspace=resume['bank_keyspace']
        else:
            if (output/'stage_A_best.pth').exists() or (output/'stage_B_best.pth').exists():
                raise FileExistsError('Training output already contains checkpoints. Use mode=resume or choose a new output_dir.')
            cfg['_run_id']=uuid.uuid4().hex
            prepare_pretrained(cfg); model=build_resroit(cfg,classes).to(cfg['device'])
        write_json(output/'resolved_config.json',cfg)
        if resume is None or resume['stage']=='A':
            a_path=train_stage(model,'A',train_records,val_records,classes,cfg,signature,None,resume)
            del model
            torch.cuda.empty_cache()
            model,_=load_checkpoint(a_path,cfg['device'])
            bank_keyspace=build_reference_bank(model,train_records,cfg)
            write_json(output/'memory_audit.json',dict(source='train-only; best Stage A raw aligned GT RoIs',keyspace=bank_keyspace,
                        entries=int(model.roi_head.memory.bank_keys.shape[0]),query_exclusion='same group excluded during training',
                        inference_updates=False,aggregation='one mean key/value per class and group'))
            resume=None
        b_path=train_stage(model,'B',train_records,val_records,classes,cfg,signature,bank_keyspace,resume)
        del model
        torch.cuda.empty_cache()
        model,b_payload=load_checkpoint(b_path,cfg['device'])
        # Deployment is the validation-selected full Stage-B model, not an ensemble.
        predictions,logs,timing=predict_records(model,val_records,cfg)
        val_metrics=evaluate_obbs(predictions,val_records,len(classes),cfg)
        save_predictions(predictions,val_records,classes,output/'validation',cfg)
        write_json(output/'validation'/'agent_log.json',logs)
        test_metrics=None
        test_records=[r for r in records if r['split']=='test']
        if cfg['evaluate_test'] and test_records:
            test_predictions,test_logs,test_timing=predict_records(model,test_records,cfg)
            test_metrics=evaluate_obbs(test_predictions,test_records,len(classes),cfg)
            save_predictions(test_predictions,test_records,classes,output/'test',cfg)
            write_json(output/'test'/'agent_log.json',test_logs)
        result=dict(mode=mode,model='SRD',classes=classes,checkpoint=str(b_path),data_audit=audit,dataset_signature=signature,
                    validation=val_metrics,test=test_metrics,timing=timing,parameters=sum(p.numel() for p in model.parameters()),
                    environment=environment,bank_keyspace=bank_keyspace)
    result['total_seconds']=time.perf_counter()-started
    result['peak_cuda_allocated_bytes']=torch.cuda.max_memory_allocated()
    write_json(output/'run_report.json',result)
    print(f'Finished. Outputs: {output}')
    return result

In [ ]:
# 执行唯一模型。结果及诊断写入 CFG["output_dir"]。
RESULT = run_model(CFG)
RESULT

## 来源与许可

检测器继承 OpenMMLab MMRotate 0.3.4 ReDet/RoITransRoIHead。`RESRoIHead.forward_train`、`RESReDet.forward_train` 在其 Apache-2.0 原流程上添加了逐级 ignore 坐标处理；设备一致 NMS、缓存隔离、参考推理及训练管线属于本次改动。Copyright (c) OpenMMLab. All rights reserved.

原始代码及论文：[MMRotate v0.3.4](https://github.com/open-mmlab/mmrotate/tree/v0.3.4)，[ReDet](https://arxiv.org/abs/2103.07733)，[RoI Transformer](https://arxiv.org/abs/1812.00155)。外部依赖各自遵循其许可证。以下保留改编部分的 Apache-2.0 许可：

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```